In [1]:
# 1. Configuration
DATA_PATH = "path/to/your/NDHS_BR_file.dta"  # Replace with actual file path
OUTDIR = "revision_results_v2"
SEED = 20261017

WINDOW_MONTHS = 60
CENSOR_MONTHS = 1
N_BOOT_DEC = 500
N_BOOT_DEC_FULL = 200
N_BOOT_CI = 1000
N_BOOT_SEQ = 300
RUN_EXTENDED = True
RUN_MULTILEVEL = True
N_QUAD = 20
FACILITY_M15_CODES = list(range(20, 40))
print("Configured successfully")

Configured successfully


In [2]:
!pip install pyreadstat

In [3]:
# 3. Helper functions
import os, re, json, math, time, warnings
import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import expit, logit
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(os.path.join(OUTDIR, "figures"), exist_ok=True)
rng_master = np.random.default_rng(SEED)

plt.rcParams.update({"font.family": "DejaVu Sans", "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight"})

NUM = {}
CHECKS = []

def put(key, val):
    NUM[key] = val

def f1(x): return f"{x:.1f}"
def f2(x): return f"{x:.2f}"
def f3(x): return f"{x:.3f}"
def pfmt(p):
    if p < 0.001: return "<0.001"
    return f"{p:.3f}"
def orci(o, lo, hi): return f"{o:.2f} ({lo:.2f}–{hi:.2f})"
def ci_txt(lo, hi, d=1): return f"{lo:.{d}f}–{hi:.{d}f}"

def wlogit(X, y, w, beta0=None, maxit=40, tol=1e-9):
    keep = w > 0
    Xk, yk, wk = X[keep], y[keep], w[keep]
    k = X.shape[1]
    if beta0 is None:
        beta = np.zeros(k)
        m = np.average(yk, weights=wk)
        beta[0] = math.log(m / (1 - m))
    else:
        beta = beta0.copy()

    def ll(b):
        eta = Xk @ b
        return np.sum(wk * (yk * eta - np.logaddexp(0, eta)))

    cur = ll(beta)
    for _ in range(maxit):
        p = expit(Xk @ beta)
        g = Xk.T @ (wk * (yk - p))
        H = (Xk * (wk * p * (1 - p))[:, None]).T @ Xk
        try:
            step = np.linalg.solve(H, g)
        except np.linalg.LinAlgError:
            step = np.linalg.lstsq(H, g, rcond=None)[0]
        t = 1.0
        while t > 1e-4:
            new = ll(beta + t * step)
            if new >= cur - 1e-10:
                break
            t /= 2
        beta = beta + t * step
        done = np.max(np.abs(t * step)) < tol
        cur = new
        if done:
            break
    return beta

class Design:

    def __init__(self, strata, psu):
        key = pd.DataFrame({"s": strata, "c": psu}).drop_duplicates().reset_index(drop=True)
        key["pid"] = np.arange(len(key))
        self.key = key
        self.n_units = len(key)
        self.unit_stratum = pd.factorize(key["s"])[0]
        self.n_strata = key["s"].nunique()
        self.df = self.n_units - self.n_strata
        sizes = pd.Series(self.unit_stratum).value_counts()
        self.lonely = int((sizes == 1).sum())
        self.members = [np.where(self.unit_stratum == h)[0] for h in range(self.n_strata)]
        self._map = key.set_index(["s", "c"])["pid"]

    def pid_of(self, strata, psu):
        idx = pd.MultiIndex.from_arrays([strata, psu])
        return self._map.reindex(idx).values.astype(int)

    def raowu(self, rng):
        mult = np.ones(self.n_units)
        for idx in self.members:
            nh = len(idx)
            if nh < 2:
                continue
            draws = rng.integers(0, nh, size=nh - 1)
            counts = np.bincount(draws, minlength=nh)
            mult[idx] = counts * nh / (nh - 1)
        return mult

def cluster_cov(S, unit, unit_stratum, n_units):
    S = np.atleast_2d(S.T).T if S.ndim == 1 else S
    k = S.shape[1]
    T = np.zeros((n_units, k))
    for j in range(k):
        T[:, j] = np.bincount(unit, weights=S[:, j], minlength=n_units)
    Tdf = pd.DataFrame(T)
    means = Tdf.groupby(unit_stratum).transform("mean").values
    nh = pd.Series(unit_stratum).map(pd.Series(unit_stratum).value_counts()).values
    scale = np.where(nh > 1, nh / np.maximum(nh - 1, 1), 0.0)
    D = (T - means) * np.sqrt(scale)[:, None]
    return D.T @ D

def make_X(D, spec):
    cols = [np.ones(len(D))]
    names = ["Intercept"]
    for item in spec:
        if item[0] == "cat":
            _, var, cats = item
            for c in cats[1:]:
                cols.append((D[var].values == c).astype(float))
                names.append(f"{var}[{c}]")
        else:
            _, var = item
            cols.append(D[var].values.astype(float))
            names.append(var)
    return np.column_stack(cols), names

class Fit:
    pass

def fit_model(D, spec, design, w=None, unit=None, unit_stratum=None, n_units=None, dfree=None):
    X, names = make_X(D, spec)
    y = D["neonatal_death"].values.astype(float)
    w = D["w"].values if w is None else w
    beta = wlogit(X, y, w)
    p = expit(X @ beta)
    A = (X * (w * p * (1 - p))[:, None]).T @ X
    S = X * (w * (y - p))[:, None]
    if unit is None:
        unit, unit_stratum, n_units, dfree = D["pid"].values, design.unit_stratum, design.n_units, design.df
    B = cluster_cov(S, unit, unit_stratum, n_units)
    Ainv = np.linalg.inv(A)
    V = Ainv @ B @ Ainv
    f = Fit()
    f.beta, f.V, f.names, f.X, f.y, f.w, f.p = beta, V, names, X, y, w, p
    f.se = np.sqrt(np.diag(V))
    f.df = dfree
    f.tcrit = stats.t.ppf(0.975, dfree)
    f.pval = 2 * stats.t.sf(np.abs(beta / f.se), dfree)
    return f

def or_rows(f, var, cats):
    out = []
    for c in cats[1:]:
        j = f.names.index(f"{var}[{c}]")
        b, se = f.beta[j], f.se[j]
        out.append(dict(var=var, cat=c, OR=math.exp(b), lo=math.exp(b - f.tcrit * se), hi=math.exp(b + f.tcrit * se), p=f.pval[j], b=b, se=se))
    return out

def wald_p(f, var, cats):
    idx = [f.names.index(f"{var}[{c}]") for c in cats[1:]]
    b = f.beta[idx]
    Vs = f.V[np.ix_(idx, idx)]
    k = len(idx)
    W = float(b @ np.linalg.solve(Vs, b))
    return float(stats.f.sf(W / k, k, max(f.df - k + 1, 1)))

def ratio_domain(y, w, mask, pid, design):
    Wd = np.sum(w * mask)
    R = np.sum(w * mask * y) / Wd
    z = w * mask * (y - R) / Wd
    V = cluster_cov(z.reshape(-1, 1), pid, design.unit_stratum, design.n_units)[0, 0]
    se = math.sqrt(V)
    if 0 < R < 1:
        sl = se / (R * (1 - R))
        t = stats.t.ppf(0.975, design.df)
        lo, hi = expit(logit(R) - t * sl), expit(logit(R) + t * sl)
    else:
        lo = hi = R
    return R * 1000, lo * 1000, hi * 1000

print("Helper functions defined.")

Helper functions defined.


In [4]:
# 4. Load the data
import pyreadstat

_, meta0 = pyreadstat.read_dta(DATA_PATH, metadataonly=True)
available = set(meta0.column_names)
required = ["b0", "b3", "b4", "b5", "b6", "b7", "b11", "bord", "v005", "v008", "v011", "v021", "v022",
            "v024", "v025", "v106", "v190", "v501"]
optional = ["caseid", "v191", "v104", "m14", "m15", "m18", "v023"]
missing_req = [c for c in required if c not in available]
if "v022" in missing_req and "v023" in available:
    required[required.index("v022")] = "v023"
    missing_req = [c for c in required if c not in available]
assert not missing_req, f"Required variables not found in file: {missing_req}"
cols = required + [c for c in optional if c in available]
raw, meta = pyreadstat.read_dta(DATA_PATH, usecols=cols, apply_value_formats=False)
STRATA_VAR = "v022" if "v022" in raw.columns else "v023"
print(f"Loaded {len(raw):,} births, {raw.shape[1]} columns. Stratum variable: {STRATA_VAR}")

labels = meta.variable_value_labels.get("v024", {})
print("\nv024 value labels in the data file:", labels)
EXPECTED = {"north central", "north east", "north west", "south east", "south south", "south west"}
zone_map = {}
for code, lab in labels.items():
    key = str(lab).strip().lower().replace("-", " ")
    for e in EXPECTED:
        if e in key:
            zone_map[code] = e.title()
OLD_MAP = {1: "North Central", 2: "North East", 3: "North West", 4: "South East", 5: "South South", 6: "South West"}
if set(zone_map.values()) != {z.title() for z in EXPECTED}:
    print("WARNING: the six zones could not be matched to the value labels; the default mapping is used.")
    zone_map = OLD_MAP
zone_fixed = any(zone_map.get(k) != v for k, v in OLD_MAP.items())
put("ZONE_FIX_SENT",
    ("Verification against the value labels in the data file showed that the codes for two geopolitical zones had been interchanged in our earlier "
     "coding. We corrected this and regenerated every zone-specific result.") if zone_fixed else
    "We re-verified the coding of geopolitical zone against the value labels in the data file and found it correct.")
print("\nZone mapping used:", zone_map)
print("Differs from the default mapping:", zone_fixed)

Loaded 104,557 births, 25 columns. Stratum variable: v022

v024 value labels in the data file: {1: 'north west', 2: 'north east', 3: 'north central', 4: 'south east', 5: 'south south', 6: 'south west'}

Zone mapping used: {1: 'North West', 2: 'North East', 3: 'North Central', 4: 'South East', 5: 'South South', 6: 'South West'}
Differs from the default mapping: True


In [5]:
# 5. Analysis variables
df = raw.copy()
df["caseid"] = df["caseid"].astype(str) if "caseid" in df.columns else np.arange(len(df)).astype(str)

for v, bad in {"v106": [9], "v501": [9], "v104": [98, 99], "m14": [98, 99], "m15": [99], "m18": [8, 9]}.items():
    if v in df.columns:
        df[v] = df[v].where(~df[v].isin(bad))

df["w"] = df["v005"] / 1e6
df["strata"] = df[STRATA_VAR]
df["psu"] = df["v021"]
df["singleton"] = ((df["b0"] == 0) | df["b0"].isna())
df["has_outcome"] = df["b5"].notna()

df["days_at_death"] = np.where(df["b6"].between(100, 199), df["b6"] - 100, np.nan)
df["nn_dhs"] = np.where(df["has_outcome"], ((df["b5"] == 0) & (df["b7"] == 0)).astype(float), np.nan)
df["neonatal_death"] = np.where(df["has_outcome"], ((df["b5"] == 0) & (df["b7"] == 0) & ~(df["days_at_death"] >= 28)).astype(float), np.nan)

df["ms"] = df["v008"] - df["b3"]
df["age_yrs"] = (df["b3"] - df["v011"]) / 12
df.loc[(df["age_yrs"] < 12) | (df["age_yrs"] > 55), "age_yrs"] = np.nan

AGE = ["15-19", "20-24", "25-29", "30-34", "35-39", "40-49"]
df["maternal_age"] = pd.cut(df["age_yrs"], [0, 20, 25, 30, 35, 40, 56], labels=AGE, right=False).astype(object)
ZONES = ["South West", "South South", "South East", "North Central", "North East", "North West"]
df["zone"] = df["v024"].map(zone_map)
df["residence"] = df["v025"].map({1: "Urban", 2: "Rural"})
df["education"] = df["v106"].map({0: "No education", 1: "Primary", 2: "Secondary", 3: "Higher"})
df["wealth"] = df["v190"].map({1: "Poorest", 2: "Poorer", 3: "Middle", 4: "Richer", 5: "Richest"})
df["wealth_trend"] = df["v190"].map({5: 0, 4: 1, 3: 2, 2: 3, 1: 4}).astype(float)
df["child_sex"] = df["b4"].map({1: "Male", 2: "Female"})
df["birth_order"] = np.select([df["bord"] == 1, df["bord"] <= 3, df["bord"] > 3], ["1st birth", "2nd-3rd birth", "4th or higher birth"], default="unknown")
df["birth_order"] = df["birth_order"].replace("unknown", np.nan)
df["birth_interval"] = np.select(
    [df["bord"] == 1, df["b11"] < 18, df["b11"] < 24, df["b11"] < 36, df["b11"] >= 36],
    ["First birth", "<18 months", "18-23 months", "24-35 months", "36+ months"], default="unknown")
df["birth_interval"] = df["birth_interval"].replace("unknown", np.nan)
df["marital"] = np.where(df["v501"].isin([1, 2]), "Currently married/in union", "Not in union").astype(object)
df.loc[df["v501"].isna(), "marital"] = np.nan
if "v104" in df.columns:
    df["stable_res"] = ((df["v104"] == 95) | ((df["v104"] < 90) & (df["v104"] * 12 >= df["ms"]))).astype(float)
    df.loc[df["v104"].isna() | df["v104"].isin([96, 97]), "stable_res"] = np.nan
else:
    df["stable_res"] = np.nan

CATS = {
    "wealth": ["Richest", "Richer", "Middle", "Poorer", "Poorest"],
    "education": ["Higher", "Secondary", "Primary", "No education"],
    "residence": ["Urban", "Rural"],
    "zone": ZONES,
    "child_sex": ["Female", "Male"],
    "birth_interval": ["36+ months", "24-35 months", "18-23 months", "<18 months", "First birth"],
    "maternal_age": AGE,
    "marital": ["Currently married/in union", "Not in union"],
}
LABEL = {"wealth": "Household wealth quintile", "education": "Maternal education", "residence": "Place of residence",
         "zone": "Geopolitical zone", "child_sex": "Sex of child", "birth_interval": "Preceding birth interval",
         "maternal_age": "Maternal age at birth (years)", "marital": "Marital status"}
MODEL_VARS = list(CATS.keys())
ALL_SPEC = [("cat", v, CATS[v]) for v in MODEL_VARS]

design = Design(df["strata"].values, df["psu"].values)
df["pid"] = design.pid_of(df["strata"].values, df["psu"].values)
print(f"Design: {design.n_units} PSUs in {design.n_strata} strata (design df = {design.df}); lonely strata = {design.lonely}")

OFFICIAL_WOMEN_SHARE = {"North Central": 18.1, "North East": 15.9, "North West": 31.8, "South East": 8.4, "South South": 11.3, "South West": 14.4}
sing = df[df["singleton"] & df["has_outcome"]]
zt = sing.groupby("zone").agg(births=("w", "size"), wsum=("w", "sum"))
zt["weighted_pct_of_births"] = 100 * zt["wsum"] / zt["wsum"].sum()
zt["official_pct_of_women"] = pd.Series(OFFICIAL_WOMEN_SHARE)
zt["ratio_births_to_women"] = zt["weighted_pct_of_births"] / zt["official_pct_of_women"]
print("\nZONE SANITY CHECK. The births/women ratio should be highest in the north (about 1.1-1.3) and lowest in the south (about 0.7-0.9).")
print(zt[["births", "weighted_pct_of_births", "official_pct_of_women", "ratio_births_to_women"]].round(2))
zt.drop(columns="wsum").round(3).to_csv(os.path.join(OUTDIR, "zone_sanity_check.csv"))

Design: 1378 PSUs in 74 strata (design df = 1304); lonely strata = 0

ZONE SANITY CHECK. The births/women ratio should be highest in the north (about 1.1-1.3) and lowest in the south (about 0.7-0.9).
               births  weighted_pct_of_births  official_pct_of_women  \
zone                                                                   
North Central   18071                   16.21                   18.1   
North East      20623                   19.10                   15.9   
North West      30605                   39.02                   31.8   
South East      11214                    7.17                    8.4   
South South     10880                    8.17                   11.3   
South West       9763                   10.33                   14.4   

               ratio_births_to_women  
zone                                  
North Central                   0.90  
North East                      1.20  
North West                      1.23  
South East                  

In [6]:
# 6. Outcome check
dead = df[(df["b5"] == 0) & (df["b7"] == 0)]
print("Deaths with b7 == 0:", len(dead))
print("Age at death in days (b6 - 100) among these deaths, selected values:")
vc = dead["days_at_death"].value_counts(dropna=False).sort_index()
print(vc.loc[[i for i in vc.index if (pd.isna(i) or i >= 25)]] if len(vc) else vc)
n_nn = int(len(dead[df.loc[dead.index, "singleton"]]))
n_ge28 = int(((dead["days_at_death"] >= 28) & df.loc[dead.index, "singleton"]).sum())
put("NN_TOTAL", f"{n_nn:,}")
put("NN_DAYS_GE28", f"{n_ge28:,}")
put("NN_PCT_GE28", f1(100 * n_ge28 / max(n_nn, 1)))
pd.crosstab(dead["days_at_death"].fillna(-1), "n").to_csv(os.path.join(OUTDIR, "outcome_check_days_at_death.csv"))
n_nodays = int(dead["days_at_death"].isna().sum())
put("NN_MISSING_B6", f"{n_nodays:,}")
put("NN_N_DAYS_28_29", f"{int(dead['days_at_death'].isin([28, 29]).sum()):,}")
print(f"Deaths with b7 == 0 and no usable b6: {n_nodays}")
print(f"Singleton deaths at 28-29 days: {n_ge28}")
assert (df.loc[df["has_outcome"], "neonatal_death"] <= df.loc[df["has_outcome"], "nn_dhs"]).all()

Deaths with b7 == 0: 3577
Age at death in days (b6 - 100) among these deaths, selected values:
days_at_death
25    7
26    3
27    2
28    9
29    7
Name: count, dtype: int64
Deaths with b7 == 0 and no usable b6: 0
Singleton deaths at 28-29 days: 14


In [7]:
# 7. Samples and flow
base = df[df["singleton"] & df["has_outcome"]].copy()
n_br, n_mult, n_miss = len(df), int((~df["singleton"]).sum()), int((df["singleton"] & ~df["has_outcome"]).sum())
full = base.copy()
win = base[(base["ms"] >= 0) & (base["ms"] <= WINDOW_MONTHS - 1)]
n_win_raw = len(win)
win_c = win[win["ms"] > CENSOR_MONTHS]
n_cens = n_win_raw - len(win_c)
cc_full = full.dropna(subset=MODEL_VARS)
cc_pri = win_c.dropna(subset=MODEL_VARS)
n_cc_drop = len(win_c) - len(cc_pri)
PRI = cc_pri.copy()
FULL = cc_full.copy()
nmr_p, nmr_p_lo, nmr_p_hi = ratio_domain(PRI["neonatal_death"].values, PRI["w"].values, np.ones(len(PRI)), PRI["pid"].values, design)

def nmr_domain(Dsub, ycol="neonatal_death"):
    pid_all = df["pid"].values
    y_all = np.zeros(len(df)); w_all = np.zeros(len(df)); m_all = np.zeros(len(df))
    idx = df.index.get_indexer(Dsub.index)
    y_all[idx] = Dsub[ycol].values; w_all[idx] = Dsub["w"].values; m_all[idx] = 1.0
    return ratio_domain(y_all, w_all, m_all, pid_all, design)

nmr_p, nmr_p_lo, nmr_p_hi = nmr_domain(PRI)
nmr_f, nmr_f_lo, nmr_f_hi = nmr_domain(FULL)
nmr_pd, nmr_pd_lo, nmr_pd_hi = nmr_domain(PRI, "nn_dhs")
nmr_fd, nmr_fd_lo, nmr_fd_hi = nmr_domain(FULL, "nn_dhs")
put("N_BR", f"{n_br:,}"); put("N_MULT", f"{n_mult:,}"); put("N_MISS_OUT", f"{n_miss:,}")
put("N_FULL_ALL", f"{len(full):,}")
put("N_WINDOW", f"{n_win_raw:,}"); put("N_CENS", f"{n_cens:,}")
put("N_PRIMARY", f"{len(PRI):,}"); put("N_PRIMARY_DEATHS", f"{int(PRI['neonatal_death'].sum()):,}")
put("N_FULL", f"{len(FULL):,}"); put("N_FULL_DEATHS", f"{int(FULL['neonatal_death'].sum()):,}")
put("N_CC_DROP", f"{n_cc_drop:,}")
put("N_MOTHERS", f"{PRI['caseid'].nunique():,}")
put("NMR_P", f1(nmr_p)); put("NMR_P_LO", f1(nmr_p_lo)); put("NMR_P_HI", f1(nmr_p_hi))
put("NMR_F", f1(nmr_f)); put("NMR_F_LO", f1(nmr_f_lo)); put("NMR_F_HI", f1(nmr_f_hi))
put("NMR_P_DHS", f1(nmr_pd)); put("NMR_P_DHS_LO", f1(nmr_pd_lo)); put("NMR_P_DHS_HI", f1(nmr_pd_hi))
put("NMR_F_DHS", f1(nmr_fd)); put("NMR_F_DHS_LO", f1(nmr_fd_lo)); put("NMR_F_DHS_HI", f1(nmr_fd_hi))
put("N_PRIMARY_DEATHS_DHS", f"{int(PRI['nn_dhs'].sum()):,}"); put("N_FULL_DEATHS_DHS", f"{int(FULL['nn_dhs'].sum()):,}")
put("MAX_YEARS", f"{int(math.ceil(FULL['ms'].max() / 12))}")
put("PCT_MISSING_CC", f1(100 * n_cc_drop / max(len(win_c), 1)))
put("NMR_P_UNW", f1(1000 * PRI["neonatal_death"].mean()))
bins = [-1, 59, 119, 179, 239, 1000]
labs = ["0–4 years", "5–9 years", "10–14 years", "15–19 years", "20+ years"]
FULL["ago"] = pd.cut(FULL["ms"], bins=bins, labels=labs)
temporal = FULL.groupby("ago", observed=True).apply(lambda g: pd.Series({
    "Births (n)": len(g), "Share of births (%)": 100 * len(g) / len(FULL),
    "NMR (weighted, per 1,000)": 1000 * np.average(g["neonatal_death"], weights=g["w"])}), include_groups=False).round(1)
_tt = temporal.copy(); _tt.index.name = "Years between birth and interview"
_tt["Births (n)"] = _tt["Births (n)"].map(lambda x: f"{int(x):,}")
_tt["Share of births (%)"] = _tt["Share of births (%)"].map(lambda x: f"{x:.1f}")
_tt["NMR (weighted, per 1,000)"] = _tt["NMR (weighted, per 1,000)"].map(lambda x: f"{x:.1f}")
_tt.to_csv(os.path.join(OUTDIR, "table_S1b_temporal_distribution.csv"))
put("TEMP_PCT_5PLUS", f1(100 - float(temporal["Share of births (%)"].iloc[0])))
put("TEMP_PCT_10PLUS", f1(float(temporal["Share of births (%)"].iloc[2:].sum())))
put("TEMP_NMR_0_4", f1(float(temporal["NMR (weighted, per 1,000)"].iloc[0])))
put("TEMP_NMR_5PLUS", f1(float(np.average(FULL.loc[FULL["ms"] >= 60, "neonatal_death"], weights=FULL.loc[FULL["ms"] >= 60, "w"]) * 1000)))
flow = pd.DataFrame([
    ("Live births in the BR file", n_br, ""),
    ("Excluded: multiple births", -n_mult, ""),
    ("Excluded: missing survival status", -n_miss, ""),
    ("Singleton births with outcome (full birth-history sample)", len(full), ""),
    ("Excluded: born more than 59 months before interview", -(len(full) - n_win_raw), ""),
    ("Singleton births 0–59 months before interview", n_win_raw, ""),
    (f"Excluded: born in the month of interview or the month before (≤ {CENSOR_MONTHS} month)", -n_cens, ""),
    ("Excluded: missing data on one or more covariates", -n_cc_drop, ""),
    ("Primary analytic sample", len(PRI), f"{int(PRI['neonatal_death'].sum()):,} neonatal deaths (0-27 days)"),
    ("Full birth-history analytic sample (complete cases)", len(FULL), f"{int(FULL['neonatal_death'].sum()):,} neonatal deaths (0-27 days)"),
], columns=["Step", "Births", "Note"])
flow["Births"] = flow["Births"].map(lambda x: f"{x:,}")
flow.to_csv(os.path.join(OUTDIR, "table_S1_flow.csv"), index=False)
print(flow.to_string(index=False))
print(f"\nPrimary NMR {nmr_p:.1f} ({nmr_p_lo:.1f}-{nmr_p_hi:.1f}); unweighted {NUM['NMR_P_UNW']}; full history {nmr_f:.1f}")

                                                                    Step  Births                              Note
                                              Live births in the BR file 104,557                                  
                                               Excluded: multiple births  -3,401                                  
                                       Excluded: missing survival status       0                                  
               Singleton births with outcome (full birth-history sample) 101,156                                  
                     Excluded: born more than 59 months before interview -74,635                                  
                           Singleton births 0–59 months before interview  26,521                                  
Excluded: born in the month of interview or the month before (≤ 1 month)    -649                                  
                        Excluded: missing data on one or more covariates       0

In [8]:
# 8. Tables 1 and 2
BI_PHRASE = {"36+ months": "intervals of 36 months or longer", "24-35 months": "an interval of 24-35 months",
             "18-23 months": "an interval of 18-23 months", "<18 months": "an interval shorter than 18 months", "First birth": "first births"}

def table1(D, tag):
    pid_all = df["pid"].values
    rows = []
    n_all = len(D)
    W_all = D["w"].sum()
    idx_all = df.index.get_indexer(D.index)
    fits_biv = {}
    for v in MODEL_VARS:
        f = fit_model(D, [("cat", v, CATS[v])], design)
        fits_biv[v] = (f, wald_p(f, v, CATS[v]))
    extra = ["birth_order"]
    for v in MODEL_VARS + extra:
        cats = CATS.get(v, ["1st birth", "2nd-3rd birth", "4th or higher birth"])
        if v == "birth_order":
            pv = np.nan
        else:
            pv = fits_biv[v][1]
        rows.append([LABEL.get(v, "Birth order"), "", "", "", "", pfmt(pv) if not np.isnan(pv) else ""])
        for c in cats:
            m = (D[v] == c).values
            y_all = np.zeros(len(df)); w_all = np.zeros(len(df)); m_all = np.zeros(len(df))
            y_all[idx_all] = D["neonatal_death"].values; w_all[idx_all] = D["w"].values; m_all[idx_all] = m.astype(float)
            r, lo, hi = ratio_domain(y_all, w_all, m_all, pid_all, design)
            rows.append(["    " + c, f"{int(m.sum()):,}", f"{int(D.loc[m, 'neonatal_death'].sum()):,}",
                         f1(100 * D.loc[m, "w"].sum() / W_all), f"{r:.1f} ({lo:.1f}–{hi:.1f})", ""])
    t = pd.DataFrame(rows, columns=["Characteristic", "Births (n)", "Neonatal deaths (n)", "Weighted % of births", "NMR per 1,000 (95% CI)", "p (design-based Wald test)"])
    return t, fits_biv

t1_pri, biv_pri = table1(PRI, "primary")
t1_pri.to_csv(os.path.join(OUTDIR, "table1_characteristics.csv"), index=False)
print(t1_pri.to_string(index=False))

def or_tables(D, fits_biv):
    f_adj = fit_model(D, ALL_SPEC, design)
    rows, flat = [], []
    for v in MODEL_VARS:
        cats = CATS[v]
        cr = or_rows(fits_biv[v][0], v, cats)
        ad = or_rows(f_adj, v, cats)
        rows.append([LABEL[v], "", "", "", "", ""])
        rows.append(["    " + cats[0] + " (reference)", "1.00", "", "1.00", "", ""])
        for c_, a_ in zip(cr, ad):
            rows.append(["    " + c_["cat"], orci(c_["OR"], c_["lo"], c_["hi"]), pfmt(c_["p"]), orci(a_["OR"], a_["lo"], a_["hi"]), pfmt(a_["p"]), ""])
            flat.append(dict(var=v, cat=c_["cat"], cOR=c_["OR"], clo=c_["lo"], chi=c_["hi"], cp=c_["p"],
                             aOR=a_["OR"], alo=a_["lo"], ahi=a_["hi"], ap=a_["p"], ab=a_["b"], ase=a_["se"]))
    t = pd.DataFrame(rows, columns=["Variable", "Crude OR (95% CI)", "p", "Adjusted OR (95% CI)", "p ", "x"]).drop(columns="x")
    t.columns = ["Characteristic", "Crude OR (95% CI)", "p", "Adjusted OR (95% CI)", "p"]
    return t, pd.DataFrame(flat), f_adj

t2_pri, flat_pri, f_adj_pri = or_tables(PRI, biv_pri)
t2_pri.to_csv(os.path.join(OUTDIR, "table2_crude_adjusted.csv"), index=False)
flat_pri.to_csv(os.path.join(OUTDIR, "flat_primary_ors.csv"), index=False)
from sklearn.metrics import roc_auc_score
auc_p = roc_auc_score(f_adj_pri.y, f_adj_pri.p, sample_weight=f_adj_pri.w)
put("AUC", f2(auc_p))
print(t2_pri.to_string(index=False)); print("Weighted AUC:", round(auc_p, 3))

                Characteristic Births (n) Neonatal deaths (n) Weighted % of births NMR per 1,000 (95% CI) p (design-based Wald test)
     Household wealth quintile                                                                                                 0.569
                       Richest      4,072                 113                 15.1       31.5 (25.3–39.2)                           
                        Richer      4,764                 173                 17.5       39.5 (32.9–47.4)                           
                        Middle      4,999                 173                 20.1       39.0 (30.5–49.9)                           
                        Poorer      5,280                 188                 22.9       36.4 (30.0–44.1)                           
                       Poorest      6,757                 245                 24.3       37.6 (31.0–45.5)                           
            Maternal education                                       

In [9]:
# 9. Result sentences
PH = {
    "zone": lambda c: f"the {c} zone",
    "education": lambda c: {"No education": "mothers with no education", "Primary": "mothers with primary education", "Secondary": "mothers with secondary education"}.get(c, c),
    "maternal_age": lambda c: f"mothers aged {c} years",
    "wealth": lambda c: f"the {c.lower()} quintile",
    "residence": lambda c: "rural births",
    "child_sex": lambda c: "boys",
    "marital": lambda c: "births to mothers not in a union",
    "birth_interval": lambda c: ("first births" if c == "First birth" else f"births after {BI_PHRASE[c]}"),
}
REFPH = {"zone": "the South West zone", "education": "mothers with higher education", "maternal_age": "mothers aged 15-19 years",
         "wealth": "the richest quintile", "residence": "urban births", "child_sex": "girls", "marital": "births to mothers in a union",
         "birth_interval": "births after intervals of 36 months or longer"}

def jn(items):
    if len(items) == 1: return items[0]
    return ", ".join(items[:-1]) + (", and " if len(items) > 2 else " and ") + items[-1]

def _cap(t): return t[:1].upper() + t[1:]

def describe(flat, var, kind="a", measure="AOR", style=0):
    sub = flat[flat["var"] == var]
    hi_, lo_, ns_ = [], [], []
    for _, r in sub.iterrows():
        o, lo, hi = (r["aOR"], r["alo"], r["ahi"]) if kind == "a" else (r["cOR"], r["clo"], r["chi"])
        txt = f"{PH[var](r['cat'])} ({measure} {o:.2f}; 95% CI {lo:.2f}\u2013{hi:.2f})"
        (hi_ if lo > 1 else lo_ if hi < 1 else ns_).append(txt)
    ref = REFPH[var]
    if style == 0:
        parts = []
        if hi_: parts.append(f"{jn(hi_)} had higher odds of neonatal death")
        if lo_: parts.append(f"{jn(lo_)} had lower odds" + ("" if hi_ else " of neonatal death"))
        if ns_: parts.append((f"{jn(ns_)} had odds that did not clearly differ" if not hi_ and not lo_ else f"for {jn(ns_)} the odds did not clearly differ"))
        out = f"Compared with {ref}, " + "; ".join(parts) + "."
    elif style == 1:
        parts = []
        if hi_: parts.append(f"higher odds of neonatal death were found for {jn(hi_)}")
        if lo_: parts.append(f"lower odds for {jn(lo_)}" if hi_ else f"lower odds of neonatal death were found for {jn(lo_)}")
        if ns_: parts.append(f"no clear difference for {jn(ns_)}" if (hi_ or lo_) else f"no clear difference in the odds of neonatal death was found for {jn(ns_)}")
        out = f"Relative to {ref}, " + jn(parts) + "."
    else:
        if hi_:
            out = f"{_cap(jn(hi_))} showed greater odds of neonatal death than {ref}"
            if lo_: out += f", whereas {jn(lo_)} showed lower odds"
            if ns_: out += f"; the odds for {jn(ns_)} did not clearly differ"
        elif lo_:
            out = f"{_cap(jn(lo_))} showed lower odds of neonatal death than {ref}"
            if ns_: out += f"; the odds for {jn(ns_)} did not clearly differ"
        else:
            out = f"The odds of neonatal death for {jn(ns_)} did not clearly differ from those for {ref}"
        out += "."
    return re.sub(r"(\d)-(\d)", "\\1\u2013\\2", out)

STYLE_OF = {"birth_interval": 0, "zone": 1, "residence": 2, "education": 2, "child_sex": 1, "maternal_age": 0, "marital": 2, "wealth": 1}
for v, key in [("birth_interval", "BI"), ("zone", "ZONE"), ("education", "EDU"), ("maternal_age", "AGE"), ("wealth", "WEALTH"),
               ("residence", "RES"), ("child_sex", "SEX"), ("marital", "MAR")]:
    put(f"DESC_{key}_ADJ", describe(flat_pri, v, "a", "AOR", STYLE_OF[v]))
    put(f"DESC_{key}_CRUDE", describe(flat_pri, v, "c", "OR", STYLE_OF[v]))

def get(flat, var, cat, f="aOR"):
    return float(flat[(flat["var"] == var) & (flat["cat"] == cat)][f].iloc[0])

def orstr(flat, var, cat, kind="a"):
    r = flat[(flat["var"] == var) & (flat["cat"] == cat)].iloc[0]
    return (f"{r['aOR']:.2f}", f"{r['alo']:.2f}", f"{r['ahi']:.2f}") if kind == "a" else (f"{r['cOR']:.2f}", f"{r['clo']:.2f}", f"{r['chi']:.2f}")

a_, l_, h_ = orstr(flat_pri, "birth_interval", "<18 months")
put("AOR_BI_LT18", a_); put("AOR_BI_LT18_LO", l_); put("AOR_BI_LT18_HI", h_)
a_, l_, h_ = orstr(flat_pri, "wealth", "Poorest", "c"); put("COR_POOREST", a_); put("COR_POOREST_LO", l_); put("COR_POOREST_HI", h_)
a_, l_, h_ = orstr(flat_pri, "wealth", "Poorest", "a"); put("AOR_POOREST", a_); put("AOR_POOREST_LO", l_); put("AOR_POOREST_HI", h_)
a_, l_, h_ = orstr(flat_pri, "residence", "Rural"); put("AOR_RURAL", a_); put("AOR_RURAL_LO", l_); put("AOR_RURAL_HI", h_)
a_, l_, h_ = orstr(flat_pri, "child_sex", "Male"); put("AOR_MALE", a_); put("AOR_MALE_LO", l_); put("AOR_MALE_HI", h_)
a_, l_, h_ = orstr(flat_pri, "education", "No education"); put("AOR_NOEDU", a_); put("AOR_NOEDU_LO", l_); put("AOR_NOEDU_HI", h_)

def sig_list(flat, var, direction=">"):
    sub = flat[flat["var"] == var]
    return [r["cat"] for _, r in sub.iterrows() if (r["alo"] > 1 if direction == ">" else r["ahi"] < 1)]

zs = sig_list(flat_pri, "zone", ">")
put("ZONES_SIG", (jn([f"the {z}" for z in zs]) + (" zones" if len(zs) > 1 else " zone")) if zs else "no zone")
put("N_ZONES_SIG", str(len(zs)))
age_hi = sig_list(flat_pri, "maternal_age", ">")
put("AGE_HI_LIST", jn(age_hi) if age_hi else "none")
age_lo = sig_list(flat_pri, "maternal_age", "<")
if any(a in age_hi for a in ["35-39", "40-49"]) and any(a in age_lo for a in ["20-24", "25-29", "30-34"]):
    put("AGE_PATTERN", "a U-shaped")
elif len(age_lo) >= 3 and not age_hi:
    put("AGE_PATTERN", "a declining")
else:
    put("AGE_PATTERN", "no clear")
edu_pt = [float(flat_pri[(flat_pri["var"] == "education") & (flat_pri["cat"] == c)]["aOR"].iloc[0]) for c in ["Secondary", "Primary", "No education"]]
put("EDU_MONO", "a graded" if edu_pt[0] <= edu_pt[1] <= edu_pt[2] else "a non-graded")
oh = [(v, r["cat"]) for v in MODEL_VARS if v != "birth_interval" for _, r in flat_pri[flat_pri["var"] == v].iterrows() if r["alo"] > 1]
ol = [(v, r["cat"]) for v in MODEL_VARS if v != "birth_interval" for _, r in flat_pri[flat_pri["var"] == v].iterrows() if r["ahi"] < 1]
put("OTHER_HIGH_LIST", jn([PH[v](c) for v, c in oh]) if oh else "no other group")
put("OTHER_LOW_LIST", jn([PH[v](c) for v, c in ol]) if ol else "no group")
put("N_OTHER_HIGH", str(len(oh)))
CHECKS += [
    ("Birth interval is the largest adjusted association (max AOR over all categories belongs to birth_interval)",
     flat_pri.loc[flat_pri["aOR"].idxmax(), "var"] == "birth_interval", "Abstract, Results para 4, Discussion para 2"),
    ("Male sex has higher odds (CI above 1)", bool(flat_pri[(flat_pri["var"] == "child_sex")]["alo"].iloc[0] > 1), "Abstract, Discussion"),
    ("Rural residence has higher odds (CI above 1)", bool(flat_pri[(flat_pri["var"] == "residence")]["alo"].iloc[0] > 1), "Abstract, Discussion"),
    ("At least one zone has higher odds than South West (CI above 1)", len(zs) > 0, "Abstract, Results, Discussion"),
    ("Every category of education below higher has OR > 1 (point estimate)", bool((flat_pri[flat_pri["var"] == "education"]["aOR"] > 1).all()), "Abstract, Discussion para on education"),
    ("Maternal age: some category at 35 years or older has higher odds than 15-19 (CI above 1)", any(a in age_hi for a in ["35-39", "40-49"]), "Abstract, Discussion para on maternal age (U-shape wording)"),
    ("Maternal age: 20-34 groups have lower odds than 15-19 (CI below 1)", all(a in sig_list(flat_pri, "maternal_age", "<") for a in ["20-24", "25-29", "30-34"]), "Discussion para on maternal age"),
    ("No adjusted wealth category has CI excluding 1", bool(((flat_pri[flat_pri["var"] == "wealth"]["alo"] <= 1) & (flat_pri[flat_pri["var"] == "wealth"]["ahi"] >= 1)).all()), "Abstract, Results wealth paragraph, Discussion wealth paragraph"),
    ("Unadjusted poorest vs richest wealth OR has CI above 1", bool(flat_pri[(flat_pri["var"] == "wealth") & (flat_pri["cat"] == "Poorest")]["clo"].iloc[0] > 1), "Abstract, Results"),
    ("Marital status adjusted CI includes 1", bool(((flat_pri[flat_pri["var"] == "marital"]["alo"] <= 1) & (flat_pri[flat_pri["var"] == "marital"]["ahi"] >= 1)).all()), "Discussion"),
]
d0 = PRI
W = d0["w"].sum()
agepct = d0.groupby("maternal_age")["w"].sum() / W * 100
put("MOD_AGE_GRP", agepct.idxmax()); put("MOD_AGE_PCT", f1(agepct.max()))
put("PCT_RURAL", f1(100 * d0.loc[d0.residence == "Rural", "w"].sum() / W))
put("PCT_NOEDU", f1(100 * d0.loc[d0.education == "No education", "w"].sum() / W))
put("PCT_POOR2", f1(100 * d0.loc[d0.wealth.isin(["Poorest", "Poorer"]), "w"].sum() / W))
put("PCT_FIRST", f1(100 * d0.loc[d0.birth_interval == "First birth", "w"].sum() / W))
put("PCT_LT18", f1(100 * d0.loc[d0.birth_interval == "<18 months", "w"].sum() / W))
zp = d0.groupby("zone")["w"].sum() / W * 100
put("ZONE_LARGEST", zp.idxmax()); put("ZONE_LARGEST_PCT", f1(zp.max()))
print("Successful")

Successful


In [10]:
# 10. NMR by group
zone_nmr = {}
for z in ZONES:
    zone_nmr[z] = nmr_domain(PRI[PRI["zone"] == z])
res_nmr = {r: nmr_domain(PRI[PRI["residence"] == r]) for r in ["Urban", "Rural"]}
HZ = max(zone_nmr, key=lambda z: zone_nmr[z][0]); LZ = min(zone_nmr, key=lambda z: zone_nmr[z][0])
put("HZ", HZ); put("LZ", LZ)
put("NMR_HZ", f1(zone_nmr[HZ][0])); put("NMR_LZ", f1(zone_nmr[LZ][0]))
put("NMR_RURAL", f1(res_nmr["Rural"][0])); put("NMR_URBAN", f1(res_nmr["Urban"][0]))
for key, (v, c) in {"LT18": ("birth_interval", "<18 months"), "GE36": ("birth_interval", "36+ months"), "MALE": ("child_sex", "Male"), "FEMALE": ("child_sex", "Female")}.items():
    put(f"NMR_{key}", f1(nmr_domain(PRI[PRI[v] == c])[0]))
sigv = [LABEL[v].lower() for v in MODEL_VARS if biv_pri[v][1] < 0.05]
nsv = [LABEL[v].lower() for v in MODEL_VARS if biv_pri[v][1] >= 0.05]
put("BIV_SIG_SENT", f"In design-based tests, {jn(sigv)} " + ("were" if len(sigv) > 1 else "was") + " associated with neonatal death (p < 0.05)" +
    (f", whereas {jn(nsv)} " + ("were" if len(nsv) > 1 else "was") + " not (Table 1)." if nsv else " (Table 1)."))
zt_nmr = pd.DataFrame([(z, *zone_nmr[z]) for z in ZONES] + [(r, *res_nmr[r]) for r in ["Urban", "Rural"]], columns=["group", "nmr", "lo", "hi"])
zt_nmr.to_csv(os.path.join(OUTDIR, "nmr_by_zone_residence.csv"), index=False)
CHECKS.append(("The zone with the highest NMR is in the north (North Central, North East or North West)", HZ.startswith("North"), "Discussion para on zones"))
CHECKS.append(("Rural NMR exceeds urban NMR", res_nmr["Rural"][0] > res_nmr["Urban"][0], "Results, Discussion"))

fig, axs = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw={"width_ratios": [3, 1.3]})
order = sorted(ZONES, key=lambda z: -zone_nmr[z][0])
nat = nmr_p
for ax, groups, data, title in [(axs[0], order, zone_nmr, "Geopolitical zone"), (axs[1], ["Rural", "Urban"], res_nmr, "Residence")]:
    vals = [data[g][0] for g in groups]; lo = [data[g][0] - data[g][1] for g in groups]; hi = [data[g][2] - data[g][0] for g in groups]
    cols_ = ["#B03A2E" if v > nat else "#2E6F95" for v in vals]
    ax.bar(range(len(groups)), vals, yerr=[lo, hi], color=cols_, capsize=3, width=0.6, edgecolor="white")
    ax.axhline(nat, color="#333", ls="--", lw=1.1)
    ax.set_xticks(range(len(groups))); ax.set_xticklabels(groups, rotation=30, ha="right", fontsize=9)
    ax.set_title(title, fontsize=10); ax.spines[["top", "right"]].set_visible(False)
axs[0].set_ylabel("Neonatal deaths per 1,000 live births (95% CI)")
axs[0].text(0.99, 0.97, f"Dashed line: analytic-sample NMR ({nat:.1f})", transform=axs[0].transAxes, ha="right", va="top", fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUTDIR, "figures", "fig2_nmr_zone_residence.png")); plt.close()
print("Figure 2 saved. Highest zone:", HZ, "Lowest zone:", LZ)

Figure 2 saved. Highest zone: North West Lowest zone: South South


In [11]:
# 11. Wealth sequence
MODELS = {
    "M1": [], "M2": ["education"], "M2g": ["zone", "residence"], "M3": ["education", "zone", "residence"],
    "M4": ["education", "zone", "residence", "maternal_age", "birth_interval", "child_sex", "marital"],
    "M4_noedu": ["zone", "residence", "maternal_age", "birth_interval", "child_sex", "marital"],
    "M4_nogeo": ["education", "maternal_age", "birth_interval", "child_sex", "marital"],
}
WEALTH_SPEC = [("cat", "wealth", CATS["wealth"])]

def seq_stats(D, w=None, beta_cache=None):
    out = {}
    w_ = D["w"].values if w is None else w
    for name, extra in MODELS.items():
        spec = WEALTH_SPEC + [("cat", v, CATS[v]) for v in extra]
        X, names = make_X(D, spec)
        y = D["neonatal_death"].values.astype(float)
        b = wlogit(X, y, w_, None if beta_cache is None else beta_cache.get(name))
        j = names.index("wealth[Poorest]")
        X1, X0 = X.copy(), X.copy()
        wcols = [i for i, n in enumerate(names) if n.startswith("wealth[")]
        X1[:, wcols] = 0; X0[:, wcols] = 0; X1[:, j] = 1
        rd = 1000 * np.sum(w_ * (expit(X1 @ b) - expit(X0 @ b))) / np.sum(w_)
        spec_t = [("num", "wealth_trend")] + [("cat", v, CATS[v]) for v in extra]
        Xt, nt = make_X(D, spec_t)
        bt = wlogit(Xt, y, w_)
        out[name] = dict(b_poor=b[j], b_all=b[[i for i in wcols]], rd=rd, b_trend=bt[1], beta=b)
    return out

base_seq = seq_stats(PRI)
cache = {k: v["beta"] for k, v in base_seq.items()}
t0 = time.time()
boot_seq = []
for r in range(N_BOOT_SEQ):
    mult = design.raowu(rng_master)
    w_b = PRI["w"].values * mult[PRI["pid"].values]
    boot_seq.append(seq_stats(PRI, w_b, cache))
    if r == 9:
        print(f"  wealth-sequence bootstrap: {(time.time() - t0) / 10 * N_BOOT_SEQ:.0f} s expected in total")

def pct_change(s, a, b):
    return 100 * (s[a]["b_poor"] - s[b]["b_poor"]) / s[a]["b_poor"]

rows = []
for name in MODELS:
    f = fit_model(PRI, WEALTH_SPEC + [("cat", v, CATS[v]) for v in MODELS[name]], design)
    ors = or_rows(f, "wealth", CATS["wealth"])
    bs = np.array([[bb[name]["b_poor"], bb[name]["rd"], bb[name]["b_trend"]] for bb in boot_seq])
    pc = np.array([pct_change(bb, "M1", name) for bb in boot_seq]) if name != "M1" else np.array([0.0])
    rd_lo, rd_hi = np.percentile(bs[:, 1], [2.5, 97.5])
    tr_lo, tr_hi = np.percentile(bs[:, 2], [2.5, 97.5])
    base_pc = pct_change(base_seq, "M1", name)
    rows.append(dict(model=name,
                     covariates="wealth" + "".join(f" + {v}" for v in MODELS[name]),
                     OR_richer=orci(ors[0]["OR"], ors[0]["lo"], ors[0]["hi"]), OR_middle=orci(ors[1]["OR"], ors[1]["lo"], ors[1]["hi"]),
                     OR_poorer=orci(ors[2]["OR"], ors[2]["lo"], ors[2]["hi"]), OR_poorest=orci(ors[3]["OR"], ors[3]["lo"], ors[3]["hi"]),
                     trend=f"{math.exp(base_seq[name]['b_trend']):.2f} ({math.exp(tr_lo):.2f}–{math.exp(tr_hi):.2f})",
                     pct_red=("" if name == "M1" else f"{base_pc:.0f} ({np.percentile(pc, 2.5):.0f} to {np.percentile(pc, 97.5):.0f})"),
                     rd=f"{base_seq[name]['rd']:.1f} ({rd_lo:.1f} to {rd_hi:.1f})",
                     p_poorest=pfmt(ors[3]["p"]), _or_poorest=ors[3]["OR"], _lo=ors[3]["lo"], _hi=ors[3]["hi"], _pc=base_pc, _rd=base_seq[name]["rd"]))
seq_df = pd.DataFrame(rows)
seq_main = seq_df[seq_df["model"].isin(["M1", "M2", "M3", "M4"])][["model", "covariates", "OR_richer", "OR_middle", "OR_poorer", "OR_poorest", "trend", "pct_red", "rd"]]
seq_main.columns = ["Model", "Covariates", "Richer OR (95% CI)", "Middle OR (95% CI)", "Poorer OR (95% CI)", "Poorest OR (95% CI)",
                    "OR per quintile step (95% CI)", "% reduction in log-odds, poorest vs M1 (95% CI)", "Risk difference, poorest minus richest, per 1,000 (95% CI)"]
seq_main.to_csv(os.path.join(OUTDIR, "table3_wealth_sequence.csv"), index=False)
alt = seq_df[seq_df["model"].isin(["M1", "M2", "M2g", "M3", "M4", "M4_noedu", "M4_nogeo"])][["model", "covariates", "OR_poorest", "trend", "pct_red", "rd"]]
alt.to_csv(os.path.join(OUTDIR, "table_S3_wealth_alternatives.csv"), index=False)
print(seq_df[["model", "OR_poorest", "trend", "pct_red", "rd"]].to_string(index=False))
r1, r2, r3, r4 = [seq_df[seq_df.model == m].iloc[0] for m in ["M1", "M2", "M3", "M4"]]
steps = {"maternal education": base_seq["M1"]["b_poor"] - base_seq["M2"]["b_poor"],
         "zone and residence": base_seq["M2"]["b_poor"] - base_seq["M3"]["b_poor"],
         "the maternal, birth-interval and child factors": base_seq["M3"]["b_poor"] - base_seq["M4"]["b_poor"]}
big = max(steps, key=lambda k: abs(steps[k]))
put("STEP_MAX", big)
put("SEQ_SENT",
    f"The odds ratio for the poorest compared with the richest quintile was {r1._or_poorest:.2f} (95% CI {r1._lo:.2f}–{r1._hi:.2f}) in the model containing wealth only (model 1), "
    f"{r2._or_poorest:.2f} ({r2._lo:.2f}–{r2._hi:.2f}) after maternal education was added (model 2), "
    f"{r3._or_poorest:.2f} ({r3._lo:.2f}–{r3._hi:.2f}) after zone and residence were added (model 3), and "
    f"{r4._or_poorest:.2f} ({r4._lo:.2f}–{r4._hi:.2f}) in the full model (model 4). "
    + (f"The log-odds for the poorest quintile were {r4._pc:.0f}% lower in model 4 than in model 1" if 0 < r4._pc < 100 else
       ("The log-odds for the poorest quintile fell to or below zero in model 4, so the crude association was fully accounted for by the added covariates" if r4._pc >= 100 else
        f"The log-odds for the poorest quintile were {abs(r4._pc):.0f}% higher in model 4 than in model 1"))
    + f", and the largest single change occurred when {big} " + ("was" if big != "the maternal, birth-interval and child factors" else "were")
    + f" added. The difference in average predicted risk between the poorest and richest quintiles was {r1._rd:.1f} per 1,000 births in model 1 and {r4._rd:.1f} per 1,000 in model 4.")
put("SEQ_RD_M1", f"{r1._rd:.1f}"); put("SEQ_RD_M4", f"{r4._rd:.1f}")
put("SEQ_OR_M1", f"{r1._or_poorest:.2f} (95% CI {r1._lo:.2f}\u2013{r1._hi:.2f})"); put("SEQ_OR_M4", f"{r4._or_poorest:.2f} (95% CI {r4._lo:.2f}\u2013{r4._hi:.2f})")
put("SEQ_PC_M4", f"{r4._pc:.0f}")
m2g = seq_df[seq_df.model == "M2g"].iloc[0]
put("SEQ_ALT_SENT", f"When zone and residence were added before maternal education, the odds ratio for the poorest quintile was {m2g._or_poorest:.2f} ({m2g._lo:.2f}–{m2g._hi:.2f}); "
    f"dropping maternal education from the full model gave {seq_df[seq_df.model == 'M4_noedu'].iloc[0]._or_poorest:.2f} and dropping zone and residence gave {seq_df[seq_df.model == 'M4_nogeo'].iloc[0]._or_poorest:.2f} (Supplementary Table S3).")
CHECKS.append(("Poorest-vs-richest log-odds is lower in M4 than in M1 (attenuation)", r4._pc > 0, "Results wealth paragraph, Discussion wealth paragraph"))

  wealth-sequence bootstrap: 175 s expected in total
   model       OR_poorest            trend             pct_red                   rd
      M1 1.20 (0.89–1.62) 1.02 (0.95–1.10)                       6.1 (-4.0 to 16.5)
      M2 0.94 (0.63–1.42) 0.97 (0.87–1.09)  132 (-1400 to 997) -2.0 (-16.5 to 13.0)
     M2g 0.78 (0.54–1.14) 0.92 (0.83–1.02) 235 (-2067 to 1431)  -8.7 (-24.5 to 5.1)
      M3 0.69 (0.43–1.11) 0.90 (0.79–1.02) 301 (-2856 to 1899) -13.5 (-33.4 to 2.1)
      M4 0.70 (0.43–1.13) 0.90 (0.79–1.02) 297 (-2903 to 1911) -13.0 (-32.5 to 2.8)
M4_noedu 0.77 (0.52–1.15) 0.91 (0.82–1.02) 242 (-2231 to 1491)  -9.0 (-25.0 to 5.2)
M4_nogeo 0.94 (0.61–1.43) 0.96 (0.87–1.09) 135 (-1385 to 1015) -2.2 (-18.0 to 13.4)


In [12]:
# 12. Concentration index
def frac_rank(rv, w):
    d = pd.DataFrame({"v": rv, "w": w})
    g = d.groupby("v")["w"].sum().sort_index()
    cum = g.cumsum() - 0.5 * g
    return d["v"].map(cum / g.sum()).values

def ci_vals(y, w, r):
    mu = np.average(y, weights=w)
    cov = np.sum(w * (y - mu) * (r - 0.5)) / np.sum(w)
    C = 2 * cov / mu
    return C, 4 * mu * C

def ci_boot(D, rank_col, n):
    y = D["neonatal_death"].values.astype(float); w0 = D["w"].values; rv = D[rank_col].values
    C0, E0 = ci_vals(y, w0, frac_rank(rv, w0))
    bc, be = [], []
    for _ in range(n):
        mult = design.raowu(rng_master)
        wb = w0 * mult[D["pid"].values]
        Cb, Eb = ci_vals(y, wb, frac_rank(rv, wb)); bc.append(Cb); be.append(Eb)
    return C0, E0, np.percentile(bc, [2.5, 97.5]), np.percentile(be, [2.5, 97.5])

PRI = PRI.copy(); PRI["wq"] = PRI["v190"]
C0, E0, Cci, Eci = ci_boot(PRI, "wq", N_BOOT_CI)
put("CI", f3(C0)); put("CI_LO", f3(Cci[0])); put("CI_HI", f3(Cci[1]))
put("CIE", f3(E0)); put("CIE_LO", f3(Eci[0])); put("CIE_HI", f3(Eci[1]))
ci_rows = [("Concentration index (Wagstaff), wealth quintile rank", f"{C0:.3f}", f"{Cci[0]:.3f} to {Cci[1]:.3f}"),
           ("Erreygers corrected index, wealth quintile rank", f"{E0:.3f}", f"{Eci[0]:.3f} to {Eci[1]:.3f}")]
if "v191" in PRI.columns and PRI["v191"].notna().all():
    Cc, Ec, Ccc, Ecc = ci_boot(PRI, "v191", max(N_BOOT_CI // 2, 200))
    ci_rows += [("Concentration index, continuous wealth score (v191) rank", f"{Cc:.3f}", f"{Ccc[0]:.3f} to {Ccc[1]:.3f}"),
                ("Erreygers corrected index, continuous wealth score rank", f"{Ec:.3f}", f"{Ecc[0]:.3f} to {Ecc[1]:.3f}")]
    put("CI_CONT", f3(Cc)); put("CI_CONT_LO", f3(Ccc[0])); put("CI_CONT_HI", f3(Ccc[1]))
pd.DataFrame(ci_rows, columns=["Measure", "Estimate", "95% CI (Rao-Wu bootstrap)"]).to_csv(os.path.join(OUTDIR, "table_ci.csv"), index=False)
CHECKS.append(("Concentration index is negative with CI excluding 0", Cci[1] < 0, "Abstract, Results, Discussion"))
r_ = frac_rank(PRI["wq"].values, PRI["w"].values)
o = np.argsort(r_, kind="stable")
cumpop = np.concatenate([[0], np.cumsum(PRI["w"].values[o]) / PRI["w"].sum()])
wd = PRI["w"].values * PRI["neonatal_death"].values
cumdead = np.concatenate([[0], np.cumsum(wd[o]) / wd.sum()])
fig, ax = plt.subplots(figsize=(5.6, 5.2))
ax.plot(cumpop, cumdead, color="#B03A2E", lw=2.2, label=f"Concentration curve (index {C0:.3f})")
ax.plot([0, 1], [0, 1], color="#444", ls="--", lw=1.1, label="Line of equality")
ax.fill_between(cumpop, cumdead, cumpop, color="#B03A2E", alpha=0.10)
ax.set_xlabel("Cumulative share of births, poorest to richest"); ax.set_ylabel("Cumulative share of neonatal deaths")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(loc="lower right", fontsize=9); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(os.path.join(OUTDIR, "figures", "fig3_concentration_curve.png")); plt.close()
print("Ran successfully")

Ran successfully


In [13]:
# 13. Decomposition
ZONE_COV = [v for v in MODEL_VARS if v not in ("zone",)]
RES_COV = [v for v in MODEL_VARS if v not in ("residence",)]

def build_X_cov(D, covs):
    X, names = make_X(D, [("cat", v, CATS[v]) for v in covs])
    return X[:, 1:], names[1:]

def decompose_core(Xc, y, w, g, ref="pooled", names=None, detail=False):
    k = Xc.shape[1]
    X1 = np.column_stack([np.ones(len(y)), Xc])
    A, B = g == 1, g == 0
    if ref == "pooled":
        b = wlogit(X1, y, w); pa = expit(X1[A] @ b); pb = expit(X1[B] @ b)
    elif ref == "groupA":
        b = wlogit(X1[A], y[A], w[A]); pa = expit(X1[A] @ b); pb = expit(X1[B] @ b)
    elif ref == "groupB":
        b = wlogit(X1[B], y[B], w[B]); pa = expit(X1[A] @ b); pb = expit(X1[B] @ b)
    elif ref == "pooled_dummy":
        X2 = np.column_stack([X1, g.astype(float)]); b = wlogit(X2, y, w)
        pa = expit(X1[A] @ b[:-1]); pb = expit(X1[B] @ b[:-1])
    Fa, Fb = np.average(pa, weights=w[A]), np.average(pb, weights=w[B])
    Ya, Yb = np.average(y[A], weights=w[A]), np.average(y[B], weights=w[B])
    gap = Ya - Yb
    expl = Fa - Fb
    out = dict(gap=gap * 1000, expl=expl * 1000, unexpl=(gap - expl) * 1000, pct=100 * expl / gap if gap else np.nan, Ya=Ya * 1000, Yb=Yb * 1000)
    if detail:
        beta = b[1:1 + k]
        xa, xb = np.average(Xc[A], axis=0, weights=w[A]), np.average(Xc[B], axis=0, weights=w[B])
        lin = beta * (xa - xb)
        contrib = expl * 1000 * lin / lin.sum() if lin.sum() != 0 else lin * 0
        out["detail"] = pd.Series(contrib, index=names)
    return out

def var_of(name):
    return name.split("[")[0]

def decomposition(D, group_mask_A, group_mask_B, covs, n_boot, label, refs=("pooled", "groupA", "groupB", "pooled_dummy")):
    sel = (group_mask_A | group_mask_B).values
    Ds = D[sel]
    g = group_mask_A.values[sel].astype(int)
    Xc, names = build_X_cov(Ds, covs)
    y = Ds["neonatal_death"].values.astype(float); w0 = Ds["w"].values; pidv = Ds["pid"].values
    res = {r: decompose_core(Xc, y, w0, g, r, names, detail=(r == "pooled")) for r in refs}
    boots = {r: [] for r in refs}
    t0 = time.time()
    for i in range(n_boot):
        mult = design.raowu(rng_master)
        wb = w0 * mult[pidv]
        for r in refs:
            if wb[g == 1].sum() == 0 or wb[g == 0].sum() == 0:
                continue
            try:
                d = decompose_core(Xc, y, wb, g, r)
                boots[r].append([d["gap"], d["expl"], d["unexpl"], d["pct"]])
            except Exception:
                pass
        if i == 4:
            print(f"  [{label}] bootstrap: ~{(time.time() - t0) / 5 * n_boot:.0f} s expected")
    ci = {r: np.percentile(np.array(boots[r]), [2.5, 97.5], axis=0) for r in refs}
    return res, ci, g.sum(), (g == 0).sum()

def pct_ci_ok(p, c):
    return bool(c[0][0] >= 0.25 * p["gap"] and (c[1][3] - c[0][3]) < 100)

NOT_REPORTED = "percentage interval not reported (gap imprecise)"

def decomp_tables(res, ci, title_a, title_b):
    p = res["pooled"]; c = ci["pooled"]; ok = pct_ci_ok(p, c)
    rows = [[title_a, "", "", ""],
            ["    Gap in NMR", f"{p['gap']:.1f}", "100", f"{c[0][0]:.1f} to {c[1][0]:.1f}"],
            ["    Explained by observed characteristics", f"{p['expl']:.1f}", f"{p['pct']:.1f}",
             f"{c[0][1]:.1f} to {c[1][1]:.1f}" + (f" ({c[0][3]:.1f}% to {c[1][3]:.1f}%)" if ok else f"; {NOT_REPORTED}")],
            ["    Not explained by observed characteristics", f"{p['unexpl']:.1f}", f"{100 - p['pct']:.1f}",
             f"{c[0][2]:.1f} to {c[1][2]:.1f}" + (f" ({100 - c[1][3]:.1f}% to {100 - c[0][3]:.1f}%)" if ok else f"; {NOT_REPORTED}")]]
    det = p["detail"].groupby([var_of(n) for n in p["detail"].index]).sum()
    det = det.reindex(det.abs().sort_values(ascending=False).index)
    rows.append(["    Contribution to the explained component, by variable", "", "", ""])
    for v, val in det.items():
        rows.append([f"        {LABEL.get(v, v)}", f"{val:.2f}", f"{100 * val / p['expl']:.1f}", ""])
    sens = []
    for r_, nm in [("pooled", "Pooled coefficients (primary)"), ("pooled_dummy", "Pooled coefficients with group indicator"),
                   ("groupA", f"Coefficients of {title_b.split(' vs ')[0]}"), ("groupB", f"Coefficients of {title_b.split(' vs ')[1]}")]:
        cc = ci[r_]
        sens.append([title_b, nm, f"{res[r_]['expl']:.1f}", f"{res[r_]['pct']:.1f}",
                     f"{cc[0][3]:.1f} to {cc[1][3]:.1f}" if pct_ci_ok(res[r_], cc) else "not reported (gap imprecise)", f"{res[r_]['unexpl']:.1f}"])
    return pd.DataFrame(rows, columns=["Component", "Per 1,000 live births", "% of component or gap", "95% CI (Rao-Wu bootstrap)"]), det, sens

zA = PRI["zone"] == HZ; zB = PRI["zone"] == LZ
print(f"Zone decomposition: {HZ} vs {LZ}")
res_z, ci_z, nA_z, nB_z = decomposition(PRI, zA, zB, ZONE_COV, N_BOOT_DEC, "zone")
rA = PRI["residence"] == "Rural"; rB = PRI["residence"] == "Urban"
print("Rural vs urban decomposition")
res_r, ci_r, nA_r, nB_r = decomposition(PRI, rA, rB, RES_COV, N_BOOT_DEC, "residence")
tz, detz, sens_z = decomp_tables(res_z, ci_z, f"Panel A. {HZ} compared with {LZ}", f"{HZ} vs {LZ}")
tr_, detr, sens_r = decomp_tables(res_r, ci_r, "Panel B. Rural compared with urban", "Rural vs Urban")
tdec = pd.concat([tz, tr_], ignore_index=True)
tdec.to_csv(os.path.join(OUTDIR, "table4_decomposition.csv"), index=False)
pd.DataFrame(sens_z + sens_r, columns=["Comparison", "Reference coefficients", "Explained per 1,000", "Explained % of gap", "95% CI for % explained", "Not explained per 1,000"]).to_csv(
    os.path.join(OUTDIR, "table_S4_decomposition_reference.csv"), index=False)
print(tdec.to_string(index=False))

def dec_tokens(prefix, res, ci, det):
    p, c = res["pooled"], ci["pooled"]
    put(f"{prefix}_GAP", f1(p["gap"])); put(f"{prefix}_EXPL", f1(p["expl"])); put(f"{prefix}_EXPL_PCT", f1(p["pct"]))
    put(f"{prefix}_EXPL_PCT_LO", f1(c[0][3])); put(f"{prefix}_EXPL_PCT_HI", f1(c[1][3]))
    put(f"{prefix}_UNEXPL", f1(p["unexpl"])); put(f"{prefix}_UNEXPL_PCT", f1(100 - p["pct"]))
    put(f"{prefix}_UNEXPL_PCT_LO", f1(100 - c[1][3])); put(f"{prefix}_UNEXPL_PCT_HI", f1(100 - c[0][3]))
    put(f"{prefix}_UNEXPL_LO", f1(c[0][2])); put(f"{prefix}_UNEXPL_HI", f1(c[1][2]))
    put(f"{prefix}_EXPL_LO", f1(c[0][1])); put(f"{prefix}_EXPL_HI", f1(c[1][1]))
    ok = pct_ci_ok(p, c)
    put(f"{prefix}_PCT_CI_OK", "yes" if ok else "no")
    put(f"{prefix}_EXPL_PCT_CI", f"95% CI {c[0][3]:.1f}%\u2013{c[1][3]:.1f}%" if ok else "the bootstrap interval for this percentage is too wide to be informative because the gap itself is imprecise")
    put(f"{prefix}_UNEXPL_PCT_CI", f"95% CI {100 - c[1][3]:.1f}%\u2013{100 - c[0][3]:.1f}%" if ok else "the bootstrap interval for this percentage is too wide to be informative because the gap itself is imprecise")
    put(f"{prefix}_GAP_LO", f1(c[0][0])); put(f"{prefix}_GAP_HI", f1(c[1][0]))
    put(f"{prefix}_EXPL_PCT_PAREN", f" (95% CI {c[0][3]:.1f}%\u2013{c[1][3]:.1f}%)" if ok else " (the interval is too wide to be informative)")
    put(f"{prefix}_UNEXPL_PCT_PAREN", f" (95% CI {100 - c[1][3]:.1f}%\u2013{100 - c[0][3]:.1f}%)" if ok else " (the interval is too wide to be informative)")
    top = det.reindex(det.abs().sort_values(ascending=False).index).head(3)
    put(f"{prefix}_TOP3", jn([f"{LABEL.get(v, v).lower()} ({100 * val / p['expl']:.0f}%)" for v, val in top.items()]))
    pcts = [res[r]["pct"] for r in ("pooled", "pooled_dummy", "groupA", "groupB")]
    put(f"{prefix}_PCT_MIN", f1(min(pcts))); put(f"{prefix}_PCT_MAX", f1(max(pcts)))

dec_tokens("Z", res_z, ci_z, detz); dec_tokens("R", res_r, ci_r, detr)
put("Z_NMR_A", f1(res_z["pooled"]["Ya"])); put("Z_NMR_B", f1(res_z["pooled"]["Yb"]))
put("R_NMR_A", f1(res_r["pooled"]["Ya"])); put("R_NMR_B", f1(res_r["pooled"]["Yb"]))
put("Z_N_A", f"{nA_z:,}"); put("Z_N_B", f"{nB_z:,}")
CHECKS.append(("Explained share exceeds 50% in both decompositions", res_z["pooled"]["pct"] > 50 and res_r["pooled"]["pct"] > 50, "Abstract, Discussion"))
CHECKS.append(("Bootstrap CI of the not-explained component excludes 0 in both decompositions", ci_z["pooled"][0][2] > 0 and ci_r["pooled"][0][2] > 0, "Results, Discussion (wording 'distinguishable from zero')"))

fig, axs = plt.subplots(1, 2, figsize=(9.2, 4.2))
for ax, res, ci, ttl in [(axs[0], res_z, ci_z, f"{HZ} vs {LZ}"), (axs[1], res_r, ci_r, "Rural vs urban")]:
    p, c = res["pooled"], ci["pooled"]
    vals = [p["expl"], p["unexpl"]]
    err = [[p["expl"] - c[0][1], p["unexpl"] - c[0][2]], [c[1][1] - p["expl"], c[1][2] - p["unexpl"]]]
    ax.bar([0, 1], vals, yerr=err, capsize=4, color=["#2E6F95", "#B03A2E"], width=0.55)
    ax.axhline(p["gap"], color="#333", ls="--", lw=1.1); ax.text(0.99, p["gap"], f"Total gap {p['gap']:.1f}", transform=ax.get_yaxis_transform(), va="bottom", ha="right", fontsize=8)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Explained by observed\ncharacteristics", "Not explained by observed\ncharacteristics"], fontsize=8.5)
    ax.set_title(ttl, fontsize=10); ax.spines[["top", "right"]].set_visible(False)
axs[0].set_ylabel("Neonatal deaths per 1,000 live births")
plt.tight_layout(); plt.savefig(os.path.join(OUTDIR, "figures", "fig4_decomposition.png")); plt.close()
print("Executed successfully")

Zone decomposition: North West vs South South
  [zone] bootstrap: ~37 s expected
Rural vs urban decomposition
  [residence] bootstrap: ~82 s expected
                                               Component Per 1,000 live births % of component or gap                                      95% CI (Rao-Wu bootstrap)
           Panel A. North West compared with South South                                                                                                           
                                              Gap in NMR                  16.2                   100                                                    6.6 to 26.4
                   Explained by observed characteristics                   5.7                  35.4                                 -1.5 to 13.3 (-15.6% to 73.3%)
               Not explained by observed characteristics                  10.5                  64.6                                  2.7 to 20.0 (26.7% to 115.6%)
    Contribution to the explai

In [14]:
# 14. Sensitivity analyses
def adj_flat(D, label, y_col="neonatal_death", cluster="design"):
    D2 = D.copy(); D2["neonatal_death"] = D2[y_col]
    if cluster == "design":
        f = fit_model(D2, ALL_SPEC, design)
    elif cluster == "psu_only":
        un = pd.factorize(D2["psu"])[0]
        f = fit_model(D2, ALL_SPEC, design, unit=un, unit_stratum=np.zeros(un.max() + 1, dtype=int), n_units=un.max() + 1, dfree=un.max())
    elif cluster == "mother":
        un = pd.factorize(D2["caseid"])[0]
        f = fit_model(D2, ALL_SPEC, design, unit=un, unit_stratum=np.zeros(un.max() + 1, dtype=int), n_units=un.max() + 1, dfree=un.max())
    rows = []
    for v in MODEL_VARS:
        for r in or_rows(f, v, CATS[v]):
            rows.append(dict(var=v, cat=r["cat"], OR=r["OR"], lo=r["lo"], hi=r["hi"], b=r["b"], se=r["se"], variant=label))
    return pd.DataFrame(rows), f

variants = {}
variants["Primary"] = adj_flat(PRI, "Primary")
variants["Full birth history"] = adj_flat(FULL, "Full birth history")
v_nc = win.dropna(subset=MODEL_VARS)
variants["No censoring exclusion"] = adj_flat(v_nc, "No censoring exclusion")
variants["Deaths in first month (DHS, b7 = 0)"] = adj_flat(PRI, "Deaths in first month (DHS, b7 = 0)", y_col="nn_dhs")
one = PRI.sort_values("ms").drop_duplicates("caseid", keep="first")
variants["One birth per woman (most recent)"] = adj_flat(one, "One birth per woman (most recent)")
if PRI["stable_res"].notna().any():
    sr = PRI[PRI["stable_res"] == 1]
    variants["Stable residence since birth"] = adj_flat(sr, "Stable residence since birth")
vt = pd.concat([v[0] for v in variants.values()], ignore_index=True)
vt["est"] = vt.apply(lambda r: orci(r["OR"], r["lo"], r["hi"]), axis=1)
wide = vt.pivot_table(index=["var", "cat"], columns="variant", values="est", aggfunc="first")
wide = wide[[c for c in variants.keys() if c in wide.columns]]
ordr = [(v, c) for v in MODEL_VARS for c in CATS[v][1:]]
wide = wide.reindex(pd.MultiIndex.from_tuples(ordr, names=["var", "cat"])).reset_index()
wide["var"] = wide["var"].map(LABEL)
wide.rename(columns={"var": "Variable", "cat": "Category"}).to_csv(os.path.join(OUTDIR, "table_S5_sensitivity_variants.csv"), index=False)
n_by = {k: len(d) for k, d in {"Primary": PRI, "Full birth history": FULL, "No censoring exclusion": v_nc, "Deaths in first month (DHS, b7 = 0)": PRI, "One birth per woman (most recent)": one}.items()}

p_, f_ = variants["Primary"][0], variants["Full birth history"][0]
cmp = p_.merge(f_, on=["var", "cat"], suffixes=("_p", "_f"))
cmp["inside"] = (cmp["OR_f"] >= cmp["lo_p"]) & (cmp["OR_f"] <= cmp["hi_p"])
cmp["absdiff"] = (cmp["b_p"] - cmp["b_f"]).abs()
put("SENS_K", str(len(cmp))); put("SENS_K_IN", str(int(cmp["inside"].sum())))
put("SENS_MED_DIFF", f"{cmp['absdiff'].median():.2f}"); put("SENS_MAX_DIFF", f"{cmp['absdiff'].max():.2f}")
s2 = cmp.assign(Variable=cmp["var"].map(LABEL), Category=cmp["cat"],
                Primary=cmp.apply(lambda r: orci(r["OR_p"], r["lo_p"], r["hi_p"]), axis=1),
                Full=cmp.apply(lambda r: orci(r["OR_f"], r["lo_f"], r["hi_f"]), axis=1))[["Variable", "Category", "Primary", "Full"]]
s2.columns = ["Variable", "Category", "Primary sample: AOR (95% CI)", "Full birth history: AOR (95% CI)"]
s2.to_csv(os.path.join(OUTDIR, "table_S2_primary_vs_full.csv"), index=False)
put("SENS_NMR_FULL_TEXT", f"The weighted NMR in the full birth-history sample was {nmr_f:.1f} per 1,000 (95% CI {nmr_f_lo:.1f}–{nmr_f_hi:.1f}).")
fig, ax = plt.subplots(figsize=(7.5, 9))
labs_ = [f"{LABEL[r['var']]}: {r['cat']}" for _, r in cmp.iterrows()]
yy = np.arange(len(cmp))[::-1]
ax.errorbar(cmp["OR_p"], yy + 0.15, xerr=[cmp["OR_p"] - cmp["lo_p"], cmp["hi_p"] - cmp["OR_p"]], fmt="o", color="#2E6F95", ms=4, lw=1, label="Primary (births 0–59 months)")
ax.errorbar(cmp["OR_f"], yy - 0.15, xerr=[cmp["OR_f"] - cmp["lo_f"], cmp["hi_f"] - cmp["OR_f"]], fmt="s", color="#B03A2E", ms=4, lw=1, label="Full birth history")
ax.axvline(1, color="#444", lw=0.8); ax.set_xscale("log"); ax.set_xticks([0.4, 0.5, 0.7, 1, 1.5, 2, 3]); ax.set_xticklabels(["0.4", "0.5", "0.7", "1", "1.5", "2", "3"]); ax.minorticks_off(); ax.set_yticks(yy); ax.set_yticklabels(labs_, fontsize=7)
ax.set_xlabel("Adjusted odds ratio (log scale)"); ax.legend(fontsize=8, loc="lower right"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(os.path.join(OUTDIR, "figures", "figS1_forest_windows.png")); plt.close()

a_des = variants["Primary"][1]
_, f_psu = adj_flat(PRI, "psu_only", cluster="psu_only")
_, f_mom = adj_flat(PRI, "mother", cluster="mother")
ix = [i for i, n in enumerate(a_des.names) if n != "Intercept"]
r_psu = a_des.se[ix] / f_psu.se[ix]
r_mom = a_des.se[ix] / f_mom.se[ix]
s6 = pd.DataFrame({"Coefficient": [a_des.names[i] for i in ix], "SE, stratified design": a_des.se[ix].round(4),
                   "SE, PSU clustering only": f_psu.se[ix].round(4), "SE, clustering on woman": f_mom.se[ix].round(4),
                   "Ratio design / PSU only": r_psu.round(3), "Ratio design / woman": r_mom.round(3)})
s6.to_csv(os.path.join(OUTDIR, "table_S6_variance_structure.csv"), index=False)
put("SE_RATIO_MOM_MED", f"{np.median(r_mom):.2f}"); put("SE_RATIO_MOM_MIN", f"{r_mom.min():.2f}")
put("SE_RATIO_PSU_MED", f"{np.median(r_psu):.2f}")
put("MAX_BIRTHS_PER_WOMAN", f"{int(PRI.groupby('caseid').size().max())}")
put("MEAN_BIRTHS_PER_WOMAN", f"{PRI.groupby('caseid').size().mean():.2f}")
put("N_ONE_BIRTH", f"{len(one):,}")
CHECKS.append(("Design-based SEs are at least as large as SEs clustered on the woman (median ratio >= 1)", float(np.median(r_mom)) >= 1.0, "Methods (repeated births paragraph)"))

Dz = PRI.copy()
for z in ZONES[1:]:
    Dz[f"zr_{z}"] = ((Dz["zone"] == z) & (Dz["residence"] == "Rural")).astype(float)
spec_i = ALL_SPEC + [("num", f"zr_{z}") for z in ZONES[1:]]
fi = fit_model(Dz, spec_i, design)
ii = [fi.names.index(f"zr_{z}") for z in ZONES[1:]]
bI = fi.beta[ii]; VI = fi.V[np.ix_(ii, ii)]
WI = float(bI @ np.linalg.solve(VI, bI)); pI = float(stats.f.sf(WI / len(ii), len(ii), design.df - len(ii) + 1))
put("P_INTERACTION", pfmt(pI) if pI < 0.001 else f"{pI:.3f}")
put("P_INTERACTION_EQ", "p " + ("< 0.001" if pI < 0.001 else f"= {pI:.3f}"))
pd.DataFrame([["Zone x residence interaction (joint Wald F test, 5 terms)", f"{WI / len(ii):.2f}", pfmt(pI)]], columns=["Test", "F", "p"]).to_csv(os.path.join(OUTDIR, "table_S6b_interaction.csv"), index=False)

bi_rng = [variants[k][0].query("var=='birth_interval' and cat=='<18 months'")["OR"].iloc[0] for k in variants]
put("SENS_BI_MIN", f2(min(bi_rng))); put("SENS_BI_MAX", f2(max(bi_rng)))
put("SENS_VARIANTS_N", str(len(variants) - 1))
print("Executed successfully")

Executed successfully


In [15]:
# 15. Full birth history decomposition
zA_f = FULL["zone"] == HZ; zB_f = FULL["zone"] == LZ
res_zf, ci_zf, _, _ = decomposition(FULL, zA_f, zB_f, ZONE_COV, N_BOOT_DEC_FULL, "zone, full history", refs=("pooled",))
res_rf, ci_rf, _, _ = decomposition(FULL, FULL["residence"] == "Rural", FULL["residence"] == "Urban", RES_COV, N_BOOT_DEC_FULL, "residence, full history", refs=("pooled",))
rows = []
for nm, res, ci in [(f"{HZ} vs {LZ}", res_zf, ci_zf), ("Rural vs Urban", res_rf, ci_rf)]:
    p, c = res["pooled"], ci["pooled"]
    rows.append([nm, f"{p['gap']:.1f}", f"{p['expl']:.1f}", f"{p['pct']:.1f}", f"{c[0][3]:.1f} to {c[1][3]:.1f}" if pct_ci_ok(p, c) else "not reported (gap imprecise)", f"{p['unexpl']:.1f}"])
pd.DataFrame(rows, columns=["Comparison", "Gap per 1,000", "Explained per 1,000", "Explained % of gap", "95% CI for % explained", "Not explained per 1,000"]).to_csv(
    os.path.join(OUTDIR, "table_S4b_decomposition_full.csv"), index=False)
t1_full, biv_full = table1(FULL, "full")
t1_full.to_csv(os.path.join(OUTDIR, "table_S2b_full_characteristics.csv"), index=False)
print("Executed successfully")

  [zone, full history] bootstrap: ~15 s expected
  [residence, full history] bootstrap: ~44 s expected
Executed successfully


In [16]:
# 16. Place of delivery and antenatal care
EXT_DONE = False
if RUN_EXTENDED and {"m15", "m14"}.issubset(df.columns):
    print("m15 labels:", meta.variable_value_labels.get("m15"))
    E = PRI.copy()
    E["facility"] = np.where(E["m15"].isin(FACILITY_M15_CODES), "Health facility", "Home or other").astype(object)
    E.loc[E["m15"].isna(), "facility"] = np.nan
    E["anc4"] = np.where(E["m14"] >= 4, "4 or more visits", "Fewer than 4 visits").astype(object)
    E.loc[E["m14"].isna(), "anc4"] = np.nan
    CATS["facility"] = ["Health facility", "Home or other"]; CATS["anc4"] = ["4 or more visits", "Fewer than 4 visits"]
    LABEL["facility"] = "Place of delivery"; LABEL["anc4"] = "Antenatal care visits"
    E = E.dropna(subset=["facility", "anc4"])
    if len(E) > 2000:
        miss_pct = 100 * (1 - len(E) / len(PRI))
        put("EXT_N", f"{len(E):,}"); put("EXT_MISS_PCT", f1(miss_pct))
        zA_e = E["zone"] == HZ; zB_e = E["zone"] == LZ
        out = []
        for nm, mA, mB, base_cov in [(f"{HZ} vs {LZ}", zA_e, zB_e, ZONE_COV), ("Rural vs Urban", E["residence"] == "Rural", E["residence"] == "Urban", RES_COV)]:
            r0, c0, _, _ = decomposition(E, mA, mB, base_cov, 200, f"ext base {nm}", refs=("pooled",))
            r1_, c1, _, _ = decomposition(E, mA, mB, base_cov + ["facility", "anc4"], 200, f"ext added {nm}", refs=("pooled",))
            for lab, r, c in [("Without service-use variables", r0, c0), ("With place of delivery and antenatal care", r1_, c1)]:
                p = r["pooled"]; cc = c["pooled"]
                out.append([nm, lab, f"{p['gap']:.1f}", f"{p['pct']:.1f}", f"{cc[0][3]:.1f} to {cc[1][3]:.1f}" if pct_ci_ok(p, cc) else "not reported (gap imprecise)", f"{p['unexpl']:.1f}"])
            put("EXT_Z_BASE_PCT" if "North" in nm or "South" in nm else "EXT_R_BASE_PCT", f1(r0["pooled"]["pct"]))
            put("EXT_Z_ADD_PCT" if "North" in nm or "South" in nm else "EXT_R_ADD_PCT", f1(r1_["pooled"]["pct"]))
        pd.DataFrame(out, columns=["Comparison", "Model", "Gap per 1,000", "Explained % of gap", "95% CI for % explained", "Not explained per 1,000"]).to_csv(
            os.path.join(OUTDIR, "table_S7_extended_decomposition.csv"), index=False)
        fe = fit_model(E, ALL_SPEC + [("cat", "facility", CATS["facility"]), ("cat", "anc4", CATS["anc4"])], design)
        rr = or_rows(fe, "facility", CATS["facility"]) + or_rows(fe, "anc4", CATS["anc4"])
        pd.DataFrame([[LABEL[r["var"]], r["cat"], orci(r["OR"], r["lo"], r["hi"]), pfmt(r["p"])] for r in rr],
                     columns=["Variable", "Category", "Adjusted OR (95% CI)", "p"]).to_csv(os.path.join(OUTDIR, "table_S7b_extended_ors.csv"), index=False)
        a_, l_, h_ = f2(rr[0]["OR"]), f2(rr[0]["lo"]), f2(rr[0]["hi"])
        put("EXT_FAC_OR", a_); put("EXT_FAC_LO", l_); put("EXT_FAC_HI", h_)
        EXT_DONE = True
print("Extended analysis run:", EXT_DONE)
if EXT_DONE:
    put("EXT_SENT", f"In an exploratory model restricted to the {NUM['EXT_N']} births with information on place of delivery and antenatal care visits, "
        f"birth in a health facility was associated with an adjusted odds ratio of {NUM['EXT_FAC_OR']} (95% CI {NUM['EXT_FAC_LO']}\u2013{NUM['EXT_FAC_HI']}) relative to birth at home or elsewhere. "
        f"Adding these two variables changed the share of the zone gap explained by observed characteristics from {NUM['EXT_Z_BASE_PCT']}% to {NUM['EXT_Z_ADD_PCT']}% and of the rural-urban gap from "
        f"{NUM['EXT_R_BASE_PCT']}% to {NUM['EXT_R_ADD_PCT']}% (Supplementary Table S7).")
else:
    put("EXT_SENT", "Place of delivery and antenatal care could not be analyzed because the required variables were not available in the file.")
zone_nmr_f = {z: nmr_domain(FULL[FULL["zone"] == z])[0] for z in ZONES}
put("HZ_FULL", max(zone_nmr_f, key=zone_nmr_f.get)); put("LZ_FULL", min(zone_nmr_f, key=zone_nmr_f.get))
CHECKS.append(("Highest- and lowest-NMR zones are the same in the full birth-history sample", NUM["HZ_FULL"] == HZ and NUM["LZ_FULL"] == LZ, "Results sensitivity paragraph, Discussion"))
print("Executed succcessfully")

m15 labels: {10: '---home---', 11: 'her home', 12: 'other home', 20: '---public sector---', 21: 'teaching hospital', 22: 'federal medical center', 23: 'general hospital', 24: 'primary health care center', 25: 'primary health clinic', 26: 'health post', 27: 'other public sector', 30: '---private medical sector---', 31: 'private hospital', 32: 'private clinic', 33: 'pharmacy', 34: 'patent and proprietary medicine vendors (ppmv)', 35: 'private doctor', 36: 'mobile clinic', 37: 'maternity center', 38: 'other private medical', 40: '---ngo medical sector---', 41: 'faith-based hospital', 42: 'faith-based clinic', 46: 'other ngo medical sector', 90: '---other---', 96: 'other'}
  [ext base North West vs South South] bootstrap: ~2 s expected
  [ext added North West vs South South] bootstrap: ~1 s expected
  [ext base Rural vs Urban] bootstrap: ~6 s expected
  [ext added Rural vs Urban] bootstrap: ~5 s expected
Extended analysis run: True
Executed succcessfully


In [17]:
# 17. Multilevel models
from scipy import optimize, integrate

def _glmm_groups(g):
    g = np.asarray(g)
    order = np.argsort(g, kind="stable")
    gs = g[order]
    starts = np.r_[0, np.flatnonzero(np.diff(gs)) + 1]
    sizes = np.diff(np.r_[starts, len(gs)])
    return order, starts, np.repeat(np.arange(len(starts)), sizes)

def glmm_logit(X, y, g, nq=20):
    order, starts, gid = _glmm_groups(g)
    X = np.asarray(X, float)[order]; y = np.asarray(y, float)[order]
    n, k = X.shape
    xq, wq = np.polynomial.hermite.hermgauss(nq)
    logw = np.log(wq) - 0.5 * math.log(math.pi)
    r2 = math.sqrt(2.0)

    def core(theta):
        s = math.exp(theta[k])
        etak = (X @ theta[:k])[:, None] + r2 * s * xq[None, :]
        S = np.add.reduceat(y[:, None] * etak - np.logaddexp(0.0, etak), starts, axis=0)
        A = S + logw[None, :]
        m = A.max(axis=1, keepdims=True)
        return etak, A, m[:, 0] + np.log(np.exp(A - m).sum(axis=1)), s

    def fun(theta):
        etak, A, lse, s = core(theta)
        resid = (y[:, None] - expit(etak)) * np.exp(A - lse[:, None])[gid]
        return -float(lse.sum()), -np.concatenate([X.T @ resid.sum(axis=1), [s * r2 * float((resid @ xq).sum())]])

    b = np.zeros(k); m0 = y.mean(); b[0] = math.log(m0 / (1 - m0))
    for _ in range(60):
        p = expit(X @ b)
        step = np.linalg.solve((X * (p * (1 - p))[:, None]).T @ X, X.T @ (y - p))
        b = b + step
        if np.max(np.abs(step)) < 1e-10: break
    eta = X @ b
    ll_plain = float(np.sum(y * eta - np.logaddexp(0, eta)))
    res = optimize.minimize(fun, np.r_[b, math.log(0.5)], jac=True, method="L-BFGS-B", bounds=[(None, None)] * k + [(-7.0, 3.0)],
                            options=dict(maxiter=1000, ftol=1e-13, gtol=1e-8))
    th = res.x
    H = np.zeros((k + 1, k + 1))
    for j in range(k + 1):
        h = 1e-5 * max(1.0, abs(th[j])); tp, tm = th.copy(), th.copy(); tp[j] += h; tm[j] -= h
        H[:, j] = (fun(tp)[1] - fun(tm)[1]) / (2 * h)
    H = (H + H.T) / 2
    try:
        se = np.sqrt(np.diag(np.linalg.inv(H)))
    except np.linalg.LinAlgError:
        se = np.full(k + 1, np.nan)
    s2 = math.exp(2 * th[k]); lr = max(0.0, 2 * (-res.fun - ll_plain))
    return dict(beta=th[:k], se=se[:k], sigma2=s2, ll=-res.fun, converged=bool(res.success), at_bound=bool(th[k] <= -6.99),
                sigma2_ci=(math.exp(2 * (th[k] - 1.96 * se[k])), math.exp(2 * (th[k] + 1.96 * se[k]))) if np.isfinite(se[k]) and th[k] > -6.99 else (np.nan, np.nan),
                lr_p=0.5 * float(stats.chi2.sf(lr, 1)) if lr > 0 else 1.0, n=n, J=len(starts), nll=lambda t: core(t)[2].sum() * -1.0, fun=fun)

def icc_latent(s2): return s2 / (s2 + math.pi ** 2 / 3)
def mor_(s2): return math.exp(math.sqrt(2 * s2) * stats.norm.ppf(0.75))

ML_DONE = False
if RUN_MULTILEVEL:
    try:
        ML = PRI.copy()
        wom = df.drop_duplicates("caseid")
        gw = wom.groupby("pid")
        comm = pd.DataFrame({
            "comm_noedu10": 10 * gw["education"].apply(lambda s: (s == "No education").sum() / max(s.notna().sum(), 1)),
            "comm_poor10": 10 * gw["wealth"].apply(lambda s: s.isin(["Poorest", "Poorer"]).sum() / max(s.notna().sum(), 1)),
            "comm_nwomen": gw.size()})
        ML = ML.join(comm, on="pid")
        yml = ML["neonatal_death"].values.astype(float); gml = ML["pid"].values
        m0 = glmm_logit(np.ones((len(ML), 1)), yml, gml, N_QUAD)
        X1, n1 = make_X(ML, ALL_SPEC)
        m1 = glmm_logit(X1, yml, gml, N_QUAD)
        SPEC2 = ALL_SPEC + [("num", "comm_noedu10"), ("num", "comm_poor10")]
        X2, n2 = make_X(ML, SPEC2)
        m2 = glmm_logit(X2, yml, gml, N_QUAD)
        un_ = pd.factorize(ML["psu"].astype(str) + "_" + ML["strata"].astype(str))[0]
        f_unw = fit_model(ML, ALL_SPEC, design, w=np.ones(len(ML)), unit=un_, unit_stratum=np.zeros(un_.max() + 1, dtype=int), n_units=un_.max() + 1, dfree=un_.max())
        try:
            sub = np.isin(gml, np.unique(gml)[:40])
            Xv_ = np.column_stack([np.ones(len(ML)), ML["comm_noedu10"].values])
            ms = glmm_logit(Xv_[sub], yml[sub], gml[sub], N_QUAD)
            sg = max(math.sqrt(m0["sigma2"]), 0.3)
            th = np.r_[m0["beta"][0], 0.05, math.log(sg)]
            osub, stt, gidd = _glmm_groups(gml[sub]); Xs_, ys_ = Xv_[sub][osub], yml[sub][osub]
            tot = 0.0
            for j in range(len(stt)):
                ix = np.where(gidd == j)[0]
                def integrand(u, ix=ix):
                    e = Xs_[ix] @ th[:-1] + u
                    return math.exp(float(np.sum(ys_[ix] * e - np.logaddexp(0, e))) - u * u / (2 * sg * sg)) / (sg * math.sqrt(2 * math.pi))
                tot += math.log(integrate.quad(integrand, -12 * sg, 12 * sg, epsabs=1e-14, epsrel=1e-12, limit=200)[0])
            ll_diff = abs(-ms["nll"](th) - tot)
            gnum = np.zeros(len(th))
            for j in range(len(th)):
                h = 1e-6; tp, tm = th.copy(), th.copy(); tp[j] += h; tm[j] -= h
                gnum[j] = (ms["nll"](tp) - ms["nll"](tm)) / (2 * h)
            g_diff = float(np.max(np.abs(gnum - ms["fun"](th)[1])))
            put("ML_VALID_LLDIFF", f"{ll_diff:.0e}"); put("ML_VALID_GRADDIFF", f"{g_diff:.0e}")
            print(f"Multilevel validation: |loglik - numerical integration| = {ll_diff:.1e}; max |analytic - numerical gradient| = {g_diff:.1e}")
        except Exception as e:
            print('Multilevel validation skipped:', repr(e)); put('ML_VALID_LLDIFF', 'not run'); put('ML_VALID_GRADDIFF', 'not run')

        def s2txt(m): return "<0.001" if m["sigma2"] < 0.0005 else f"{m['sigma2']:.3f}"
        def s2ci(m): return f"{m['sigma2_ci'][0]:.3f}–{m['sigma2_ci'][1]:.3f}" if np.isfinite(m["sigma2_ci"][0]) else "not estimable"
        def lrp(m): return "p < 0.001" if m["lr_p"] < 0.001 else f"p = {m['lr_p']:.3f}"
        def pcv(m): return 100 * (m0["sigma2"] - m["sigma2"]) / m0["sigma2"] if m0["sigma2"] > 0 else np.nan
        def chg(p):
            return "no change" if not np.isfinite(p) else (f"a reduction of {p:.0f}%" if p >= 0 else f"an increase of {abs(p):.0f}%")
        def pcv_cell(m): return "" if m0["sigma2"] < 0.0005 else f"{pcv(m):.0f}"
        rows = []
        for lab, m, nm in [("Null model", m0, "null"), ("Model A1: covariates in Table 2", m1, "A1"), ("Model A2: A1 plus cluster-level composition", m2, "A2")]:
            rows.append([lab, s2txt(m), s2ci(m), f"{100 * icc_latent(m['sigma2']):.1f}", f"{mor_(m['sigma2']):.2f}",
                         "" if nm == "null" else pcv_cell(m), lrp(m), f"{m['n']:,}", f"{m['J']:,}"])
        pd.DataFrame(rows, columns=["Model", "Between-cluster variance", "95% CI", "Intraclass correlation (%)", "Median odds ratio", "Change in variance vs null (%)",
                                    "LR test of zero variance", "Births", "Clusters"]).to_csv(os.path.join(OUTDIR, "table_S8_multilevel_variance.csv"), index=False)
        pri_flat = variants["Primary"][0].set_index(["var", "cat"])
        out = []; inside = 0; k_tot = 0; diffs_unw = []; diffs_geo = []
        for v in MODEL_VARS:
            for c in CATS[v][1:]:
                nm_ = f"{v}[{c}]"
                b1, s1 = m1["beta"][n1.index(nm_)], m1["se"][n1.index(nm_)]
                b2, s2_ = m2["beta"][n2.index(nm_)], m2["se"][n2.index(nm_)]
                bu = f_unw.beta[f_unw.names.index(nm_)]; su = f_unw.se[f_unw.names.index(nm_)]
                pr = pri_flat.loc[(v, c)]
                k_tot += 1
                inside += int(pr["lo"] <= math.exp(b1) <= pr["hi"])
                diffs_unw.append(abs(b1 - bu))
                if v in ("zone", "residence"): diffs_geo.append(abs(b2 - b1))
                out.append([LABEL[v], c, orci(pr["OR"], pr["lo"], pr["hi"]), orci(math.exp(bu), math.exp(bu - 1.96 * su), math.exp(bu + 1.96 * su)),
                            orci(math.exp(b1), math.exp(b1 - 1.96 * s1), math.exp(b1 + 1.96 * s1)), orci(math.exp(b2), math.exp(b2 - 1.96 * s2_), math.exp(b2 + 1.96 * s2_))])
        comm_txt = {}
        for key, lab in [("comm_noedu10", "Women with no education, cluster share (per 10 pp)"), ("comm_poor10", "Women in poorest two quintiles, cluster share (per 10 pp)")]:
            b_, s_ = m2["beta"][n2.index(key)], m2["se"][n2.index(key)]
            comm_txt[key] = orci(math.exp(b_), math.exp(b_ - 1.96 * s_), math.exp(b_ + 1.96 * s_))
            out.append(["Cluster-level composition", lab, "", "", "", comm_txt[key]])
        pd.DataFrame(out, columns=["Variable", "Category", "Primary: weighted, design-based", "Single-level, unweighted", "Random intercept (A1)", "Random intercept plus composition (A2)"]).to_csv(
            os.path.join(OUTDIR, "table_S8b_multilevel_ors.csv"), index=False)
        b1_bi = m1["beta"][n1.index("birth_interval[<18 months]")]; s1_bi = m1["se"][n1.index("birth_interval[<18 months]")]
        for nm_, m, key in [("0", m0, "0"), ("1", m1, "1"), ("2", m2, "2")]:
            put(f"ML_S2_{key}", s2txt(m)); put(f"ML_S2_{key}_CI", s2ci(m)); put(f"ML_ICC_{key}", f"{100 * icc_latent(m['sigma2']):.1f}"); put(f"ML_MOR_{key}", f"{mor_(m['sigma2']):.2f}")
        put("ML_CHG_1", chg(pcv(m1))); put("ML_CHG_2", chg(pcv(m2)))
        put("ML_LRP_0", lrp(m0)); put("ML_LRP_1", lrp(m1))
        put("ML_NOEDU_OR", comm_txt["comm_noedu10"]); put("ML_POOR_OR", comm_txt["comm_poor10"])
        put("ML_K", str(k_tot)); put("ML_K_IN", str(inside)); put("ML_MAXDIFF_UNW", f2(max(diffs_unw))); put("ML_GEO_MAXDIFF", f2(max(diffs_geo)))
        put("ML_BI_LT18", orci(math.exp(b1_bi), math.exp(b1_bi - 1.96 * s1_bi), math.exp(b1_bi + 1.96 * s1_bi)))
        put("ML_N", f"{m1['n']:,}"); put("ML_J", f"{m1['J']:,}"); put("ML_NQ", str(N_QUAD))
        put("ML_NW_MED", f"{int(ML.drop_duplicates('pid')['comm_nwomen'].median())}")
        put("ML_CORR_COMM", f"{np.corrcoef(ML['comm_noedu10'], ML['comm_poor10'])[0, 1]:.2f}")
        ok_all = all(m["converged"] and not m["at_bound"] for m in (m0, m1, m2))
        CHECKS.append(("Random-intercept models converged with a variance estimate away from zero", ok_all, "Results (multilevel sentence), Discussion"))
        def vtxt(m, null=False):
            if m["sigma2"] < 0.0005:
                return f"the variance was estimated at essentially zero (<0.001; median odds ratio {mor_(m['sigma2']):.2f})"
            core_ = f"the variance was {s2txt(m)} (median odds ratio {mor_(m['sigma2']):.2f})"
            return core_ if null or m0["sigma2"] < 0.0005 else f"{core_}, {chg(pcv(m))} relative to the null model"
        put("ML_SENT_RES",
            f"In two-level random-intercept models fitted without weights (Supplementary Table S8), the variance of the cluster intercept in the null model was {s2txt(m0)} (95% CI {s2ci(m0)}), "
            f"equivalent to an intraclass correlation of {NUM['ML_ICC_0']}% and a median odds ratio of {NUM['ML_MOR_0']}. After adjustment for the covariates in Table 2, {vtxt(m1)}. "
            f"After the cluster share of women with no education and the cluster share in the poorest two wealth quintiles were also added, {vtxt(m2)}; the odds ratios per 10-percentage-point increase "
            f"in these two shares were {NUM['ML_NOEDU_OR']} and {NUM['ML_POOR_OR']}, respectively. The odds ratios in the adjusted random-intercept model differed from those in the unweighted single-level model "
            f"by at most {NUM['ML_MAXDIFF_UNW']} in log-odds, and {NUM['ML_K_IN']} of {NUM['ML_K']} lay within the 95% confidence interval of the design-based estimates. "
            f"The zone and residence odds ratios changed by at most {NUM['ML_GEO_MAXDIFF']} in log-odds when the cluster-level measures were added.")
        ML_DONE = True
    except Exception as e:
        print("MULTILEVEL STEP FAILED:", repr(e))
print("Multilevel sensitivity analysis run:", ML_DONE)
put("ML_DONE", "yes" if ML_DONE else "no")
if not ML_DONE:
    put("ML_SENT_RES", "The multilevel sensitivity analysis was not run.")

Multilevel validation: |loglik - numerical integration| = 2.6e-08; max |analytic - numerical gradient| = 1.2e-08
Multilevel sensitivity analysis run: True


In [18]:
# 18. Validation and outputs
try:
    import statsmodels.api as sm
    Xv, nv = make_X(PRI, ALL_SPEC)
    g = sm.GLM(PRI["neonatal_death"].values, Xv, family=sm.families.Binomial(), freq_weights=PRI["w"].values).fit()
    maxdiff = float(np.max(np.abs(g.params - f_adj_pri.beta)))
    print(f"Validation: max |beta(this code) - beta(statsmodels GLM, weighted)| = {maxdiff:.2e}")
    put("VALIDATION_MAXDIFF", f"{maxdiff:.1e}")
except Exception as e:
    print("statsmodels validation skipped:", e)

_Xv, _ = make_X(PRI, ALL_SPEC)
_yv = PRI["neonatal_death"].values.astype(float); _wv = PRI["w"].values
_bs = np.array([wlogit(_Xv, _yv, _wv * design.raowu(rng_master)[PRI["pid"].values]) for _ in range(100)])
_rr = f_adj_pri.se / _bs.std(axis=0, ddof=1)
put("VALID_RATIO_MED", f"{np.median(_rr):.2f}"); put("VALID_RATIO_MIN", f"{_rr.min():.2f}"); put("VALID_RATIO_MAX", f"{_rr.max():.2f}")
print(f"Binder SE / Rao-Wu bootstrap SE: median {np.median(_rr):.2f} (range {_rr.min():.2f}-{_rr.max():.2f})")
put("LONELY_STRATA", str(design.lonely)); put("N_PSU", f"{design.n_units:,}"); put("N_STRATA", str(design.n_strata)); put("DESIGN_DF", str(design.df))
put("B_CI", f"{N_BOOT_CI:,}"); put("B_DEC", f"{N_BOOT_DEC:,}"); put("B_SEQ", f"{N_BOOT_SEQ:,}")
put("N_ONE_BIRTH", f"{len(one):,}")
put("EXT_DONE", "yes" if EXT_DONE else "no")
with open(os.path.join(OUTDIR, "manuscript_numbers.json"), "w") as fh:
    json.dump(NUM, fh, indent=1)
pd.DataFrame([(c[0], "PASS" if c[1] else "CHECK THE TEXT", c[2]) for c in CHECKS], columns=["Assumption made in the manuscript text", "Result", "Where it appears"]).to_csv(
    os.path.join(OUTDIR, "assumption_checks.csv"), index=False)
print("\nASSUMPTION CHECKS")
for c in CHECKS:
    print(("  PASS   " if c[1] else "  CHECK  ") + c[0] + f"   [{c[2]}]")
print(f"\nDone. {len(NUM)} manuscript numbers written to {OUTDIR}/manuscript_numbers.json")

Validation: max |beta(this code) - beta(statsmodels GLM, weighted)| = 2.30e-14
Binder SE / Rao-Wu bootstrap SE: median 1.02 (range 0.91-1.12)

ASSUMPTION CHECKS
  PASS   Birth interval is the largest adjusted association (max AOR over all categories belongs to birth_interval)   [Abstract, Results para 4, Discussion para 2]
  PASS   Male sex has higher odds (CI above 1)   [Abstract, Discussion]
  CHECK  Rural residence has higher odds (CI above 1)   [Abstract, Discussion]
  CHECK  At least one zone has higher odds than South West (CI above 1)   [Abstract, Results, Discussion]
  PASS   Every category of education below higher has OR > 1 (point estimate)   [Abstract, Discussion para on education]
  CHECK  Maternal age: some category at 35 years or older has higher odds than 15-19 (CI above 1)   [Abstract, Discussion para on maternal age (U-shape wording)]
  CHECK  Maternal age: 20-34 groups have lower odds than 15-19 (CI below 1)   [Discussion para on maternal age]
  PASS   No adjusted we

In [19]:
# 19. Save results
import shutil
from IPython.display import FileLink
shutil.make_archive(OUTDIR, "zip", OUTDIR)
FileLink(OUTDIR + ".zip")

C:\Users\DELL\revision_results_v2.zip